# ESP Failure Prediction & Optimal Frequency — SIC Capstone

Predicts Electrical Submersible Pump (ESP) failures from 1-minute SCADA data, identifies the developing failure cause, and selects a safe operating frequency.

**Everything runs here — no data to download.** Step 3 generates the SCADA data from scratch.

## Scope

Four **degradation-driven** failure causes: `GAS_LOCK`, `UNDERLOAD`, `HIGH_TEMP`, `HIGH_DISCHARGE`.

Two causes were deliberately taken out of scope on field-engineering grounds:
- **LOW_VOLTAGE** — a grid sag is an external supply event the well data cannot anticipate, and the VSD's undervoltage protection already handles it.
- **VIBRATION** — not a trip cause in itself, it is a condition indicator. Kept as a *sensor input* (removing the channel was tested and measurably hurt gas-lock classification), just not modelled as a failure label.

16 wells × 6 months × 1 minute = 4.19M rows. Wells 8, 9, 10 are held out entirely.

## Runtime

~12 min on a free Colab CPU (Step 3 ~5 min, Step 4 ~6 min). Step 5 adds ~4 min. Run cells in order.

## Running on your own computer

Open this notebook from inside the project folder (VS Code or Jupyter) and run the cells in order. It uses the project's `data/` folder, which already holds the dataset, the equipment specs and the trained models, so Step 3 skips the 5-minute data generation. Use the same Python that trained the saved models (3.12 on the team laptop).

## Step 0 — Install

In [ ]:
%pip install -q xgboost
import os, sys, subprocess
# One working folder for every step:
#   on Colab   -> /content/esp_multiwell   (starts empty; Step 3 generates the data)
#   on your PC -> the project's own data/ folder, which already holds the dataset and the
#                 trained models, so nothing has to be downloaded or regenerated.
IN_COLAB = "google.colab" in sys.modules
if "PROJECT" not in globals():
    _nb = globals().get("__vsc_ipynb_file__") or os.path.abspath("ESP_Capstone_Colab.ipynb")
    PROJECT = os.path.dirname(os.path.dirname(_nb))      # the folder above 1_run_in_colab/
WORK = "/content/esp_multiwell" if IN_COLAB else os.path.join(PROJECT, "data")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
os.environ["ESP_DATA"] = WORK
# charts and result files go to the project's 4_charts/latest_run/ and 3_results/latest_run/;
# the official results one level up (the Colab run the report quotes) are never overwritten
CHART_DIR = WORK if IN_COLAB else os.path.join(PROJECT, "4_charts", "latest_run")
RESULT_DIR = WORK if IN_COLAB else os.path.join(PROJECT, "3_results", "latest_run")
for _d in (CHART_DIR, RESULT_DIR):
    os.makedirs(_d, exist_ok=True)
os.environ["ESP_CHARTS"], os.environ["ESP_RESULTS"] = CHART_DIR, RESULT_DIR
if WORK not in sys.path:
    sys.path.insert(0, WORK)                             # so Step 5 can import the generator

def run_script(*args):
    """Run a project script with this notebook's own Python and print what it says."""
    env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1")
    p = subprocess.run([sys.executable, *args], cwd=WORK, env=env,
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    print("\n".join(l for l in (p.stdout + p.stderr).splitlines()
                    if "Warning" not in l and "warnings.warn" not in l and "model_persistence" not in l))
    if p.returncode != 0:
        raise RuntimeError(f"{args[0]} failed (exit {p.returncode})")

print("data and models:", WORK)
print("charts  ->", CHART_DIR)
print("results ->", RESULT_DIR)

## Step 1 — Well roster

The only file you edit to add a well. Add a row and re-run from Step 2.

In [ ]:
wells_config = '''well_id,causes,split
1,GAS_LOCK:1.0,train
2,UNDERLOAD:1.0,train
3,HIGH_TEMP:1.0,train
4,HIGH_DISCHARGE:1.0,train
5,GAS_LOCK:1.0,train
6,UNDERLOAD:1.0,train
7,GAS_LOCK:0.5;HIGH_TEMP:0.5,train
8,UNDERLOAD:0.4;HIGH_TEMP:0.6,test
9,HIGH_DISCHARGE:0.5;GAS_LOCK:0.5,test
10,GAS_LOCK:0.3;HIGH_DISCHARGE:0.3;UNDERLOAD:0.4,test
11,UNDERLOAD:1.0,train
12,HIGH_TEMP:1.0,train
13,HIGH_DISCHARGE:1.0,train
14,HIGH_TEMP:1.0,train
15,HIGH_DISCHARGE:1.0,train
16,GAS_LOCK:1.0,train
'''
with open('wells_config.csv','w') as f:
    f.write(wells_config)
import pandas as pd
print(pd.read_csv('wells_config.csv').to_string(index=False))

## Step 2 — Equipment specs and safe Hz band

Safe band = intersection of Motor, Protector and Pump rated ranges.

In [ ]:
"""
ESP equipment configuration per well: Motor, Protector, Gas Separator, Pump.

These are REPRESENTATIVE/TYPICAL specs for a training exercise (not pulled
from a real manufacturer catalog) - realistic in shape and magnitude for a
standard 5.13"-housing ESP string, used to derive a per-well SAFE OPERATING
FREQUENCY BAND (Min Hz / Max Hz), per the instructor's feedback that the
recommended frequency must respect real equipment limits, not just look
statistically optimal.

Safe band logic (intersection of component-rated ranges):
  - Motor: nameplate rated Hz (60), plus a manufacturer-derated max Hz
    (running faster than this risks winding/bearing overheat) and a min Hz
    (below this, motor cooling by fluid flow-by becomes inadequate).
  - Protector: thrust-bearing capacity implies a max Hz (higher Hz -> more
    axial thrust from the pump stages); expressed here directly as a max Hz.
  - Pump: manufacturer curve valid range (min/max Hz the pump stages are
    rated for) - going outside it risks upthrust/downthrust damage.
  - Gas Separator: type + efficiency does not change the Hz band directly,
    but wells with a less effective (or no) gas separator are more
    susceptible to gas lock at a given drawdown - noted per well.

Well Safe Hz Band = [max of all component mins, min of all component maxs]

Well roster (well_id, cause profile, train/test split) now lives in
wells_config.csv, the single shared source of truth for both this script
and generate_multiwell_data.py. To add a well, edit that file - see
HOW_TO_ADD_A_WELL.md.
"""
import numpy as np
import pandas as pd

rng = np.random.default_rng(77)

MOTOR_CATALOG = [
    # (model, rated_hp, rated_v, rated_a, min_hz, max_hz)
    ("HS 456 Series 562 Motor", 456, 2450, 68, 35, 70),
    ("HS 562 Series 562 Motor", 562, 2790, 72, 35, 75),
    ("HD 375 Series 456 Motor", 375, 2100, 61, 30, 68),
    ("HT 500 Series 562 Motor", 500, 2600, 70, 35, 72),
]

PROTECTOR_CATALOG = [
    # (model, thrust_rating_lb, max_hz)
    ("Tandem Protector - Labyrinth/Bag, 2500 lb", 2500, 68),
    ("Tandem Protector - Positive Seal, 3200 lb", 3200, 72),
    ("Single Protector - Labyrinth, 1800 lb", 1800, 65),
]

GAS_SEPARATOR_CATALOG = [
    # (model, type, efficiency_pct)
    ("Rotary Gas Separator, RGS-538", "Rotary", 85),
    ("Reverse-Flow Gas Separator, RFS-400", "Reverse-Flow", 65),
    ("None (natural separation only)", "None", 25),
]

PUMP_CATALOG = [
    # (model, stage_type, n_stages, min_hz, max_hz, bep_hz)
    ("Mixed-Flow Pump, GC-2700, 92 stages", "Mixed-Flow", 92, 40, 65, 52),
    ("Mixed-Flow Pump, GC-3300, 78 stages", "Mixed-Flow", 78, 38, 67, 54),
    ("Radial-Flow Pump, D-1750, 130 stages", "Radial-Flow", 130, 35, 63, 50),
    ("Mixed-Flow Pump, GC-4000, 65 stages", "Mixed-Flow", 65, 40, 70, 55),
]


def load_wells_config(path="wells_config.csv"):
    cfg = pd.read_csv(path)
    causes_by_well = {}
    for _, row in cfg.iterrows():
        pairs = str(row["causes"]).split(";")
        causes_by_well[int(row["well_id"])] = [p.split(":")[0] for p in pairs]
    return causes_by_well


WELL_CAUSES = load_wells_config()

rows = []
for well_id, causes in WELL_CAUSES.items():
    motor = MOTOR_CATALOG[rng.integers(len(MOTOR_CATALOG))]
    protector = PROTECTOR_CATALOG[rng.integers(len(PROTECTOR_CATALOG))]
    pump = PUMP_CATALOG[rng.integers(len(PUMP_CATALOG))]

    # Wells with GAS_LOCK in their cause profile are deliberately given a
    # weaker gas separator (or none) - this is WHY they're gas-lock-prone,
    # tying the equipment config to the failure mode rather than being arbitrary
    if "GAS_LOCK" in causes:
        gas_sep = GAS_SEPARATOR_CATALOG[rng.choice([1, 2])]  # Reverse-Flow or None
    else:
        gas_sep = GAS_SEPARATOR_CATALOG[rng.choice([0, 1], p=[0.7, 0.3])]  # mostly good Rotary

    min_hz = max(motor[4], pump[3])
    max_hz = min(motor[5], protector[2], pump[4])

    rows.append({
        "well_id": well_id,
        "motor_model": motor[0], "motor_rated_hp": motor[1], "motor_rated_v": motor[2], "motor_rated_a": motor[3],
        "motor_min_hz": motor[4], "motor_max_hz": motor[5],
        "protector_model": protector[0], "protector_thrust_rating_lb": protector[1], "protector_max_hz": protector[2],
        "gas_separator_model": gas_sep[0], "gas_separator_type": gas_sep[1], "gas_separator_efficiency_pct": gas_sep[2],
        "pump_model": pump[0], "pump_stage_type": pump[1], "pump_n_stages": pump[2],
        "pump_min_hz": pump[3], "pump_max_hz": pump[4], "pump_bep_hz": pump[5],
        "safe_min_hz": min_hz, "safe_max_hz": max_hz,
        "dominant_causes": ", ".join(causes),
    })

specs = pd.DataFrame(rows).sort_values("well_id").reset_index(drop=True)

# sanity check: every well needs a valid non-empty band, and BEP should fall inside it
assert (specs["safe_min_hz"] < specs["safe_max_hz"]).all(), "invalid band for some well"
specs["bep_in_band"] = specs.apply(lambda r: r["pump_bep_hz"] >= r["safe_min_hz"] and r["pump_bep_hz"] <= r["safe_max_hz"], axis=1)
print(specs[["well_id", "safe_min_hz", "safe_max_hz", "pump_bep_hz", "bep_in_band", "gas_separator_type", "gas_separator_efficiency_pct"]].to_string(index=False))

specs.to_csv("well_equipment_specs.csv", index=False)
print(f"\nSaved well_equipment_specs.csv ({len(specs)} wells)")


## Step 3 — Simulate SCADA data  ⏱ ~5 min

Each well runs a state machine (normal → onset → shutdown → ramp) with physics-inspired sensors.

**Two physics corrections were made here after measurement showed the model could not possibly detect two of the four causes:**
- `UNDERLOAD` was an instantaneous random event with zero precursor. Real inflow starvation develops gradually as the fluid level falls — that is why pump-off controllers exist. Now accumulates. Catch rate **6% → 59%**.
- `HIGH_DISCHARGE` had a 12 psi precursor against 15 psi of sensor noise — mathematically undetectable. Raised to a realistic amplitude for scale build-up. Catch rate **10% → 55%**.

Drawdown also now responds to pump rate, so slowing the pump genuinely relieves underload — without that, frequency control could not work in the simulation.

In [ ]:
%%writefile generate_multiwell_data.py
"""
Multi-well ESP SCADA simulation, 6 months each, 1-minute resolution,
with MIXED failure causes across wells (not all wells face the same
failure mode) - matching the team's real-data plan.

Well roster (which wells exist, their cause profile, train/test split) is
read from wells_config.csv - the single shared config also used by
equipment_specs.py. To add a well (e.g. more data to test against later),
edit that file and re-run this script - see HOW_TO_ADD_A_WELL.md. This
script is INCREMENTAL: it only simulates well_ids that are in
wells_config.csv but not yet in the output CSV, then appends them, so
adding one well is fast - it does not re-simulate everything.

Failure causes modeled (each well is assigned 1-2 dominant causes,
so the dataset has a realistic mix, and some wells are "healthy/rare
failure" wells):

  1. GAS_LOCK        - gas interference at intake; LOW current, erratic
                        pressure; frequency-related (reducing Hz helps)
  2. UNDERLOAD       - pump off-design / low fluid supply; LOW current;
                        NOT frequency-related in the same way - often a
                        supply/inflow issue, not fixed by changing Hz
  3. HIGH_TEMP       - inadequate cooling / high ambient + high load;
                        motor temp climbs steadily; frequency-related
                        (running harder raises heat)
  4. HIGH_DISCHARGE  - downstream restriction (scale/choke); discharge
                        pressure climbs, current climbs; frequency-related
  5. VIBRATION       - mechanical wear / misalignment; vibration climbs
                        independent of frequency choice; NOT solved by
                        changing frequency alone (mechanical issue)
  6. LOW_VOLTAGE     - surface power supply sag; voltage drops, current
                        becomes unstable; NOT frequency-related at all -
                        an electrical supply problem; this one is an
                        abrupt external-grid event, not a gradual
                        degradation, so (realistically) it has little to
                        no leading indicator - see precursor leak below.

PRECURSOR LEAK (new): previously, sensors carried literally zero signal
before the state machine flipped from "normal" to "onset" - detection had
no genuine early-warning signal to find for a meaningful chunk of the
60-minute pre-trip window, which was part of why the detector's recall was
weak. GAS_LOCK, HIGH_TEMP, HIGH_DISCHARGE and VIBRATION each have a latent
degradation index that was already accumulating during "normal" state
(gas_accum, thermal_load, fouling_index, wear_index) but never leaked into
the sensors until onset officially began. Now a small, physically-motivated
fraction of each index leaks into its associated raw sensor throughout
"normal" state too, growing as the index approaches its trigger threshold -
representing the gradual degradation a real SCADA feed would actually show.
UNDERLOAD and LOW_VOLTAGE are sporadic/external triggers with no
accumulator, so they intentionally keep no precursor - that is a realistic
difference between a wearing-out failure and a sudden-onset one, not a bug.

Output: one CSV with a `well_id`, `split` (train/test) column, a
`failure_cause` label on the exact trip-minute row, and an `onset_cause`
label on every row while that well is actively in the "onset" state (the
clean, non-ambiguous signal Stage 2 root-cause classification trains on).
"""
import os
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)

MINUTES_PER_DAY = 24 * 60
N_DAYS = 182
N = N_DAYS * MINUTES_PER_DAY

CAUSES = ["GAS_LOCK", "UNDERLOAD", "HIGH_TEMP", "HIGH_DISCHARGE", "VIBRATION", "LOW_VOLTAGE"]
FREQ_RELATED = {"GAS_LOCK": True, "UNDERLOAD": False, "HIGH_TEMP": True,
                 "HIGH_DISCHARGE": True, "VIBRATION": False, "LOW_VOLTAGE": False}

OUT_DIR = os.environ.get("ESP_DATA", os.path.dirname(os.path.abspath(__file__)))
OUT_PATH = f"{OUT_DIR}/multiwell_scada_10wells_6mo.csv"
CONFIG_PATH = f"{OUT_DIR}/wells_config.csv"


def load_well_profiles(path=CONFIG_PATH):
    cfg = pd.read_csv(path)
    profiles = {}
    for _, row in cfg.iterrows():
        pairs = str(row["causes"]).split(";")
        causes = {}
        for p in pairs:
            name, weight = p.split(":")
            causes[name] = float(weight)
        profiles[int(row["well_id"])] = {"causes": causes, "split": row["split"]}
    return profiles


WELL_PROFILES = load_well_profiles()

P_RES = 2600.0
K_DRAWDOWN = 9.0
# Pump capacity per Hz (bbl/d/Hz), sized so that at a typical well's
# best-efficiency point the pump is roughly matched to what the reservoir can
# deliver. This is what makes "optimal frequency" a real question: past the
# inflow limit, extra pump speed produces nothing - it only pulls the fluid
# level down faster and brings the next pump-off trip forward.
C_PUMP = 15.0
K_BOOST = 13.5

# Loaded from well_equipment_specs.csv (run equipment_specs.py first) - each
# well's operating point and reduced-frequency restart point are now derived
# from ITS OWN equipment-rated safe band, not a shared global constant.
_specs = pd.read_csv(f"{OUT_DIR}/well_equipment_specs.csv").set_index("well_id")


def simulate_well(well_id, causes_profile, seed, controller=None):
    """controller(well_id, minute, freq, pi, current, safe_min, safe_max, bep) -> commanded Hz.

    Optional closed-loop frequency control. It sees only what a SCADA feed exposes -
    the current frequency and sensor readings - never the simulator's latent state.
    Used to back-test the underload frequency recommendation against the same well
    and the same random seed with control off.
    """
    local_rng = np.random.default_rng(seed)
    rows = []
    shutdown_events = []

    spec = _specs.loc[well_id]
    SAFE_MIN_HZ = float(spec["safe_min_hz"])
    SAFE_MAX_HZ = float(spec["safe_max_hz"])
    FREQ_BASE = float(spec["pump_bep_hz"])              # normal operating point = pump's best-efficiency Hz
    FREQ_RESTART = SAFE_MIN_HZ + 2.0                     # reduced-freq restart, just above the equipment floor
    gas_sep_efficiency = float(spec["gas_separator_efficiency_pct"])
    # weaker gas separator -> faster gas accumulation for wells prone to GAS_LOCK
    gas_lock_multiplier = (100.0 - gas_sep_efficiency) / 75.0

    reservoir_pressure = local_rng.uniform(1800, 3200)
    productivity_index = local_rng.uniform(0.8, 2.5)
    baseline_supply_voltage = local_rng.uniform(4100, 4200)

    state = "normal"
    freq = FREQ_BASE
    gas_accum = local_rng.uniform(0.05, 0.15)
    wear_index = local_rng.uniform(0.05, 0.15)      # mechanical wear (vibration cause)
    thermal_load = local_rng.uniform(0.05, 0.15)    # cumulative thermal stress (high temp cause)
    fouling_index = local_rng.uniform(0.05, 0.15)   # discharge restriction buildup
    drawdown_index = local_rng.uniform(0.05, 0.15)  # fluid level falling toward the pump (underload)
    onset_minutes = 0
    shutdown_minutes = 0
    shutdown_target = None
    ramp_minutes = 0
    ramp_duration = None
    current_cause = None
    commanded_freq = FREQ_BASE      # what a controller has asked for (BEP when uncontrolled)
    Tm, Ti = 178.0, 182.0

    def days_to_next_onset():
        return local_rng.uniform(3.0, 9.0) * MINUTES_PER_DAY

    accel_rate = 1.0 / days_to_next_onset()

    for i in range(N):
        cause = current_cause

        # ---------------- state machine ----------------
        if state == "normal":
            freq = commanded_freq
            gas_accum += accel_rate * gas_lock_multiplier * (freq / FREQ_BASE) ** 2 * local_rng.normal(1.0, 0.1) * 0.5
            wear_index += accel_rate * local_rng.normal(1.0, 0.15) * 0.35
            thermal_load += accel_rate * local_rng.normal(1.0, 0.15) * 0.35
            fouling_index += accel_rate * local_rng.normal(1.0, 0.15) * 0.35
            # Drawdown builds only while the pump outruns the well's inflow, so it must
            # depend on pump rate. Affinity law: Q proportional to frequency, so demand
            # scales with (f/f_bep)^2 against a fixed inflow capacity. At the BEP the
            # pump slightly outruns inflow and the level falls; roughly 10-15% slower and
            # the level recovers. Without this term, reducing frequency could not fix
            # underload in the simulation - which is the whole point of a pump-off control.
            _pump_demand = (freq / FREQ_BASE) ** 2
            drawdown_index += accel_rate * 0.35 * ((_pump_demand - 0.80) / 0.20) * local_rng.normal(1.0, 0.15)
            gas_accum, wear_index = max(gas_accum, 0), max(wear_index, 0)
            thermal_load, fouling_index = max(thermal_load, 0), max(fouling_index, 0)
            drawdown_index = max(drawdown_index, 0)

            voltage_sag = local_rng.random() < 0.0001  # rare abrupt supply sag event
            triggers = {
                "GAS_LOCK": gas_accum >= 1.0,
                # PHYSICS CORRECTION: underload was modelled as an instantaneous random
                # event, which made it unpredictable by construction (0-sigma precursor,
                # 6% catch rate). Real inflow starvation / pump-off develops gradually as
                # the fluid level over the pump falls - which is exactly why pump-off
                # controllers exist. It now accumulates like the other degradation modes.
                "UNDERLOAD": drawdown_index >= 1.0,
                "HIGH_TEMP": thermal_load >= 1.0,
                "HIGH_DISCHARGE": fouling_index >= 1.0,
                "VIBRATION": wear_index >= 1.0,
                "LOW_VOLTAGE": voltage_sag,
            }
            active = [c for c in CAUSES if triggers[c] and c in causes_profile]
            if active:
                weights = np.array([causes_profile[c] for c in active])
                current_cause = local_rng.choice(active, p=weights / weights.sum())
                state = "onset"
                onset_minutes = 0

        elif state == "onset":
            freq = commanded_freq
            onset_minutes += 1
            p_trip = min(0.02 * onset_minutes, 0.9)
            if local_rng.random() < p_trip:
                state = "shutdown"
                shutdown_minutes = 0
                shutdown_target = local_rng.uniform(90, 240)
                shutdown_events.append((i, current_cause))

        elif state == "shutdown":
            freq = 0.0
            shutdown_minutes += 1
            if shutdown_minutes >= shutdown_target:
                state = "ramp"
                ramp_minutes = 0
                ramp_duration = local_rng.uniform(300, 600)
                freq = FREQ_RESTART
                # reset the specific stressor that caused this trip
                if current_cause == "GAS_LOCK":
                    gas_accum = local_rng.uniform(0.05, 0.15)
                elif current_cause == "VIBRATION":
                    wear_index = local_rng.uniform(0.05, 0.15)
                elif current_cause == "HIGH_TEMP":
                    thermal_load = local_rng.uniform(0.05, 0.15)
                elif current_cause == "HIGH_DISCHARGE":
                    fouling_index = local_rng.uniform(0.05, 0.15)
                elif current_cause == "UNDERLOAD":
                    drawdown_index = local_rng.uniform(0.05, 0.15)
                accel_rate = 1.0 / days_to_next_onset()

        elif state == "ramp":
            ramp_minutes += 1
            frac = min(ramp_minutes / ramp_duration, 1.0)
            freq = FREQ_RESTART + frac * (commanded_freq - FREQ_RESTART)
            if ramp_minutes >= ramp_duration:
                state = "normal"
                current_cause = None

        onset_severity = min(onset_minutes / 40.0, 1.0) if state == "onset" else 0.0
        c = cause if state in ("onset", "shutdown") else None

        # ---------------- precursor leak (pre-onset, gradual causes only) ----------------
        # A small, physically-motivated fraction of each latent index leaks
        # into its sensor even during "normal" state, growing as the index
        # approaches its trigger threshold (>=1.0). Gives detection a real
        # early-warning signal instead of a flat baseline until the instant
        # onset officially begins. UNDERLOAD/LOW_VOLTAGE are sporadic/
        # external triggers with no accumulator - intentionally no precursor.
        precursor_current_dip = 0.05 * min(gas_accum, 1.3)              # GAS_LOCK precursor
        precursor_vibration = 0.10 * min(wear_index, 1.3)                # VIBRATION precursor
        precursor_tm = 3.0 * min(thermal_load, 1.3)                      # HIGH_TEMP precursor
        # Scale/fouling raises discharge pressure by tens of psi as it builds - the
        # previous 12 psi sat below the 15 psi sensor noise, i.e. 0.8 sigma, which is
        # why HIGH_DISCHARGE was effectively invisible (10% catch rate).
        precursor_pd = 55.0 * min(fouling_index, 1.3)                    # HIGH_DISCHARGE precursor
        # Falling fluid level: intake pressure declines and the starved pump draws less
        # current. Both are standard pump-off indicators on a real SCADA feed.
        precursor_pi_drop = 45.0 * min(drawdown_index, 1.3)              # UNDERLOAD precursor
        precursor_cur_sag = 0.04 * min(drawdown_index, 1.3)              # UNDERLOAD precursor

        # ---------------- sensor synthesis ----------------
        if state == "shutdown":
            current = max(local_rng.normal(0.2, 0.1), 0)
            voltage = max(local_rng.normal(30, 15), 0)
            vibration = max(local_rng.normal(0.02, 0.01), 0)
            pi = P_RES - 5 * (1 - min(shutdown_minutes / shutdown_target, 1.0)) + local_rng.normal(0, 8)
            pd_ = max(local_rng.normal(40, 10), 0)
            tm_target = 90 + 90 * (1 - min(shutdown_minutes / 200, 1.0))
            ti_target = 184 + 2 * min(shutdown_minutes / 200, 1.0)
        else:
            base_current = 29 + 0.52 * freq
            base_voltage = baseline_supply_voltage
            vib_base = 0.14 + 0.004 * (freq - 40) + 0.5 * wear_index if wear_index > 1 else 0.14 + 0.004 * (freq - 40)
            pi_penalty = 0.0
            pd_extra = 0.0
            efficiency = 1.0
            volt_drop = 0.0
            current_mult = 1.0

            if c == "GAS_LOCK":
                dip = onset_severity * rng.uniform(0.15, 0.45)
                current_mult *= (1 - dip)
                vibration_extra = onset_severity * rng.uniform(0.6, 1.8)
                pi_penalty = (max(gas_accum - 0.6, 0) ** 1.5) * 55 + onset_severity * rng.uniform(0, 90)
                efficiency = 1 - 0.55 * onset_severity * rng.uniform(0.7, 1.0)
            elif c == "UNDERLOAD":
                dip = onset_severity * rng.uniform(0.25, 0.55)
                current_mult *= (1 - dip)
                pi_penalty = onset_severity * rng.uniform(40, 120)
                efficiency = 1 - 0.3 * onset_severity
                vibration_extra = onset_severity * rng.uniform(0.05, 0.2)
            elif c == "HIGH_TEMP":
                vibration_extra = onset_severity * rng.uniform(0.1, 0.3)
                current_mult *= (1 + 0.05 * onset_severity)
                efficiency = 1 - 0.1 * onset_severity
            elif c == "HIGH_DISCHARGE":
                pd_extra = onset_severity * rng.uniform(150, 400) + fouling_index * 30 if fouling_index > 1 else onset_severity * rng.uniform(150, 400)
                current_mult *= (1 + 0.15 * onset_severity)
                vibration_extra = onset_severity * rng.uniform(0.1, 0.3)
                efficiency = 1 - 0.2 * onset_severity
            elif c == "VIBRATION":
                vibration_extra = onset_severity * rng.uniform(1.0, 2.5) + max(wear_index - 1, 0) * 0.8
                current_mult *= (1 + 0.03 * onset_severity)
                efficiency = 1 - 0.1 * onset_severity
            elif c == "LOW_VOLTAGE":
                volt_drop = onset_severity * rng.uniform(300, 900)
                current_mult *= (1 + 0.35 * onset_severity)  # current climbs to compensate
                vibration_extra = onset_severity * rng.uniform(0.05, 0.15)
                efficiency = 1 - 0.15 * onset_severity
            else:
                vibration_extra = 0.0

            # precursor leak applies whenever the well is actually running
            # (normal/onset/ramp), on top of any onset-specific effect above
            current_mult *= (1 - precursor_current_dip) * (1 - precursor_cur_sag)
            vibration_extra += precursor_vibration
            pd_extra += precursor_pd

            current = base_current * current_mult + local_rng.normal(0, 0.9)
            voltage = base_voltage - volt_drop + local_rng.normal(0, 12)
            vibration = max(vib_base + vibration_extra + local_rng.normal(0, 0.05), 0)

            pi = P_RES - K_DRAWDOWN * freq - pi_penalty - precursor_pi_drop + local_rng.normal(0, 12)
            pd_ = pi + K_BOOST * freq * efficiency + pd_extra + local_rng.normal(0, 15)

            tm_target = (165 + 0.85 * freq + onset_severity * (25 if c == "HIGH_TEMP" else 10)
                         + max(thermal_load - 1, 0) * 15 + precursor_tm)
            ti_target = 182 + onset_severity * 3 + local_rng.normal(0, 0.3)

        Tm += (tm_target - Tm) * 0.02 + local_rng.normal(0, 0.15)
        Ti += (ti_target - Ti) * 0.05 + local_rng.normal(0, 0.1)

        # ---------------- production ----------------
        # Inflow performance: what the reservoir can deliver at this drawdown.
        # Pump capacity: affinity law, proportional to speed.
        # Actual rate is the lesser of the two - a pump cannot produce fluid the
        # reservoir does not supply.
        if state == "shutdown":
            production_bpd = 0.0
        else:
            q_pump = C_PUMP * freq
            q_inflow = productivity_index * max(P_RES - pi, 0.0)
            production_bpd = min(q_pump, q_inflow)

        if controller is not None and state in ("normal", "onset"):
            commanded_freq = float(np.clip(
                controller(well_id=well_id, minute=i, freq=freq, pi=pi, current=current,
                           safe_min=SAFE_MIN_HZ, safe_max=SAFE_MAX_HZ, bep=FREQ_BASE),
                SAFE_MIN_HZ, SAFE_MAX_HZ))

        rows.append((
            well_id, i, state, c if c else "", round(freq, 2), round(current, 2), round(voltage, 1),
            round(Ti, 2), round(Tm, 2), round(pi, 1), round(pd_, 1), round(vibration, 3),
            round(production_bpd, 1),
        ))

    df = pd.DataFrame(rows, columns=[
        "well_id", "minute", "state", "onset_cause", "frequency_hz", "motor_current_a", "voltage_v",
        "Ti_intake_temp_f", "Tm_motor_temp_f", "Pi_intake_psi", "Pd_discharge_psi", "vibration_g",
        "production_bpd",
    ])
    df["shutdown_event"] = 0
    df["failure_cause"] = ""
    for idx, cause in shutdown_events:
        df.loc[idx, "shutdown_event"] = 1
        df.loc[idx, "failure_cause"] = cause
    df["safe_min_hz"] = SAFE_MIN_HZ
    df["safe_max_hz"] = SAFE_MAX_HZ
    return df, shutdown_events


# ============================================================
# Incremental generation: only simulate wells missing from the output CSV
# ============================================================
existing_well_ids = set()
existing_df = None
if os.path.exists(OUT_PATH):
    existing_df = pd.read_csv(OUT_PATH, usecols=["well_id"])
    existing_well_ids = set(existing_df["well_id"].unique().tolist())
    del existing_df

wells_to_simulate = {wid: p for wid, p in WELL_PROFILES.items() if wid not in existing_well_ids}

if not wells_to_simulate:
    print("No new wells to simulate - wells_config.csv already fully generated in", OUT_PATH)
else:
    print(f"Simulating {len(wells_to_simulate)} well(s) not yet in the dataset: {sorted(wells_to_simulate)}")
    new_dfs = []
    summary = []
    for well_id, profile in wells_to_simulate.items():
        df_well, events = simulate_well(well_id, profile["causes"], seed=1000 + well_id)
        df_well["split"] = profile["split"]
        start = pd.Timestamp("2026-01-01 00:00:00")
        df_well["timestamp"] = start + pd.to_timedelta(df_well["minute"], unit="m")
        new_dfs.append(df_well)
        cause_counts = pd.Series([c for _, c in events]).value_counts().to_dict()
        summary.append({"well_id": well_id, "split": profile["split"], "n_trips": len(events), **cause_counts})
        print(f"well {well_id:2d} [{profile['split']}] causes={list(profile['causes'].keys())}: {len(events)} trips -> {cause_counts}")

    new_full = pd.concat(new_dfs, ignore_index=True)
    new_full = new_full[[
        "well_id", "split", "timestamp", "state", "frequency_hz", "motor_current_a", "voltage_v",
        "Ti_intake_temp_f", "Tm_motor_temp_f", "Pi_intake_psi", "Pd_discharge_psi", "vibration_g",
        "shutdown_event", "failure_cause", "onset_cause", "safe_min_hz", "safe_max_hz",
        "production_bpd",
    ]]

    # QA: confirm no simulated frequency ever violated its own well's safe band
    violations = new_full[(new_full["frequency_hz"] > 0) &
                           ((new_full["frequency_hz"] < new_full["safe_min_hz"] - 0.01) |
                            (new_full["frequency_hz"] > new_full["safe_max_hz"] + 0.01))]
    print(f"\nSafe-band QA (new wells): {len(violations)} rows violate the well's equipment-rated Hz band "
          f"(0 expected; freq=0 rows during shutdown are excluded from this check).")

    if os.path.exists(OUT_PATH):
        new_full.to_csv(OUT_PATH, mode="a", header=False, index=False)
        print(f"\nAppended {len(new_full):,} rows to existing:", OUT_PATH)
    else:
        new_full.to_csv(OUT_PATH, index=False)
        print(f"\nSaved {len(new_full):,} rows to new file:", OUT_PATH)

    print("\nSummary (new wells only):")
    print(pd.DataFrame(summary).fillna(0))

full = pd.read_csv(OUT_PATH, usecols=["well_id", "split", "shutdown_event"])
print("\nFull dataset now covers", full["well_id"].nunique(), "wells,", len(full), "rows")
print(full.groupby(["well_id", "split"])["shutdown_event"].sum().rename("n_trips"))


In [ ]:
# Colab starts empty, so the data is always generated there (~5 min).
# Locally, data/ already holds the dataset: the generator finds all 16 wells and stops.
# Set REGENERATE_DATA = True to rebuild it from scratch; the simulator is seeded, so the
# result is identical.
REGENERATE_DATA = IN_COLAB
_csv = os.path.join(WORK, "multiwell_scada_10wells_6mo.csv")
if REGENERATE_DATA and os.path.exists(_csv):
    os.remove(_csv)          # the generator only adds wells missing from the file, so start clean
run_script("generate_multiwell_data.py")

## Step 4 — Full analysis  ⏱ ~6 min

Unsupervised branch, both supervised stages, baselines, alarm policy, all charts.

Key design decisions, all measured rather than assumed:
- **Leakage prevention** — split by well *and* by time. Model selection uses validation only.
- **Selection on event-level cost**, not minute-level ROC-AUC, and a catch only counts if it arrives ≥30 min ahead — otherwise the system is detecting failures rather than predicting them.
- **Per-well adaptive thresholds** — a single global cutoff does not transfer between wells.
- **Alarm persistence** — 15 consecutive minutes above threshold, and false alarms counted as *events*, not minutes.

In [ ]:
"""
Full ESP analysis on the 10-well mixed-cause, equipment-aware dataset:

  UNSUPERVISED branch: K-Means / DBSCAN clustering of operating states
    (independent of the supervised branch - per the team's diagram: two
    parallel branches, unsupervised does NOT feed the supervised model).

  SUPERVISED branch, two stages:
    Stage 1: binary classification - will a trip happen in the next H
             minutes?
    Stage 2: given a likely failure, which cause? (multi-class)

Key fix vs. the previous run: features are now Z-SCORE NORMALIZED PER WELL
before modeling. Each well has its own baseline sensor levels (different
reservoir pressure, productivity index, supply voltage - see
generate_multiwell_data.py), so raw/absolute feature values don't transfer
across wells. Normalizing against each well's OWN baseline is what lets a
model trained on train wells generalize to fully unseen wells. For train
wells, the baseline is computed from the FIT period only (no time leakage).
For held-out test wells, the baseline is computed from that well's own
first 30 days (a realistic "calibration window" after a new well starts
producing - you don't get another well's baseline in the field either).

SECOND fix (this run): Stage 2 previously trained on every row inside the
Stage-1 H=60-minute "trip within H" window, with the cause back-filled from
the eventual shutdown event. A lot of that window is still statistically
normal - the fault hasn't visibly started yet - so a large share of the
"positive" training rows carried a cause label with no matching signal in
the features, which is what collapsed the classifier onto the single
best-represented class (LOW_VOLTAGE). Stage 2 now trains and evaluates only
on rows where the simulator's own state machine says the well is actively
in "onset" for a given cause (the `onset_cause` column) - a clean,
non-ambiguous label instead of a noisy one.

THIRD fix (this run): the dataset previously gave detection zero real
lead-time signal - sensors were flat until the instant onset began. A small
precursor leak (see generate_multiwell_data.py) now lets GAS_LOCK,
HIGH_TEMP, HIGH_DISCHARGE and VIBRATION's underlying degradation indices
show up gradually beforehand; a longer 240-minute rolling mean is added
below so the model has a feature that can actually see that drift.
UNDERLOAD and LOW_VOLTAGE stay abrupt/external by design (a real supply
sag has little lead time either), so they're expected to stay harder to
catch early - that's a realistic difference between failure modes, not a
gap to hide.

More wells (16 total, 13 train / 3 held-out test) give every cause at
least 2 training wells instead of 1, for better cross-well generalization.
Add wells any time via wells_config.csv - see HOW_TO_ADD_A_WELL.md.

Equipment integration (per instructor feedback: "define the operationally
allowable frequency range"): each well's SAFE_MIN_HZ / SAFE_MAX_HZ (from
well_equipment_specs.csv, derived from Motor/Protector/Pump ratings) is
carried through and used to (a) confirm the simulated data never violates
it [checked in generate_multiwell_data.py], and (b) clip any operating
frequency recommendation to stay within the specific well's own equipment
limits - never a shared global range.

Leakage prevention:
  - Well-based split: wells 8, 9, 10 held out entirely (never in training).
  - Time-based split: within the 7 training wells, final 6 weeks held out
    as a validation window.

Baselines: fixed-threshold rule (mirrors current SCADA alarm logic) and
last-value/persistence.

Reported: lead time, false alarms, missed failures, production impact,
alongside standard model metrics (ROC-AUC, F1, precision/recall).
"""
import warnings
warnings.filterwarnings("ignore")
import gc

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
)
from xgboost import XGBClassifier

OUT = WORK
H = 30  # failure horizon in minutes - shorter horizon + run-life feature combined (best result)
CAUSE_NAMES = ["GAS_LOCK", "UNDERLOAD", "HIGH_TEMP", "HIGH_DISCHARGE", "VIBRATION", "LOW_VOLTAGE"]

df = pd.read_csv(f"{OUT}/multiwell_scada_10wells_6mo.csv", parse_dates=["timestamp"])
df = df.sort_values(["well_id", "timestamp"]).reset_index(drop=True)
df = df.drop(columns=["Ti_intake_temp_f"])  # not used as a modeling feature - drop early to save memory
specs = pd.read_csv(f"{OUT}/well_equipment_specs.csv").set_index("well_id")

TRAIN_WELLS = sorted(df.loc[df["split"] == "train", "well_id"].unique())
TEST_WELLS = sorted(df.loc[df["split"] == "test", "well_id"].unique())
print("Train wells (fully seen):", TRAIN_WELLS)
print("Test wells (fully held out):", TEST_WELLS)

# Sensor channels fed to the models. DROP_SENSORS lets a run exclude channels
# without editing the pipeline - used to test the field engineer's argument that
# vibration and supply voltage are mostly alarm load: vibration does not itself
# trip an ESP (it is a condition indicator, not a trip cause) and a surface
# voltage sag is an external grid event that well data cannot anticipate.
import os
SENSOR_COLS = ["motor_current_a", "voltage_v", "vibration_g", "Pi_intake_psi", "Pd_discharge_psi", "Tm_motor_temp_f"]
DROP_SENSORS = [s for s in os.environ.get("DROP_SENSORS", "").split(",") if s]
if DROP_SENSORS:
    SENSOR_COLS = [c for c in SENSOR_COLS if c not in DROP_SENSORS]
    print("Dropped sensor channels:", DROP_SENSORS)
print("Sensor channels in use:", SENSOR_COLS)
RESULTS_NAME = os.environ.get("RESULTS_NAME", "results.txt")
for col in SENSOR_COLS + ["frequency_hz"]:
    df[col] = df[col].astype("float32")

# ============================================================
# Rolling features, per well - computed via groupby().transform() so new
# columns are added onto the existing dataframe in place, rather than
# building a full per-well copy of every column for every well (which is
# what blew past the memory budget once the dataset grew to 16 wells).
# ============================================================
RAW_FEATURES = ["frequency_hz", "minutes_since_last_shutdown"] + SENSOR_COLS
ROLL_STATS = ["roll15_mean", "roll15_std", "roll60_mean", "roll60_std", "roll240_mean", "chg5"]
ROLL_FEATURES = [f"{c}_{stat}" for c in SENSOR_COLS for stat in ROLL_STATS]
ALL_FEATURES = RAW_FEATURES + ROLL_FEATURES

grouped = df.groupby("well_id", sort=False)
for col in SENSOR_COLS:
    df[f"{col}_roll15_mean"] = grouped[col].transform(lambda s: s.rolling(15, min_periods=1).mean()).astype("float32")
    df[f"{col}_roll15_std"] = grouped[col].transform(lambda s: s.rolling(15, min_periods=1).std()).fillna(0).astype("float32")
    df[f"{col}_roll60_mean"] = grouped[col].transform(lambda s: s.rolling(60, min_periods=1).mean()).astype("float32")
    df[f"{col}_roll60_std"] = grouped[col].transform(lambda s: s.rolling(60, min_periods=1).std()).fillna(0).astype("float32")
    df[f"{col}_roll240_mean"] = grouped[col].transform(lambda s: s.rolling(240, min_periods=1).mean()).astype("float32")
    df[f"{col}_chg5"] = grouped[col].transform(lambda s: s.diff(5)).fillna(0).astype("float32")
    gc.collect()
del grouped
gc.collect()
print("Rolling features done.")

# ============================================================
# RELATIVE (SELF-REFERENCED) FEATURES - TESTED AND TURNED OFF.
#
# Hypothesis: Stage 1 on RAW absolute levels scored ~0.82 ROC-AUC on
# validation (held-out weeks of wells it trained on) but only ~0.66 on
# fully unseen wells. Each well has its own reservoir pressure,
# productivity index, supply voltage and BEP frequency, so an absolute
# threshold like "intake pressure below 2100 psi means trouble" looked like
# memorization of a specific well rather than a transferable rule. The fix
# would be features that compare each sensor ONLY against its own recent
# history, leaving no absolute level to memorize. (An earlier attempt at
# this failed for a different reason - the detrended features were ADDED
# alongside the raw ones, leaving the absolute values still available to
# overfit on - so this version REPLACES them, which is the test the idea
# deserved.)
#
# RESULT: the gap did close, but from the wrong end. Validation ROC-AUC fell
# 0.82 -> 0.685 while test only moved 0.66 -> 0.63, and the event-level catch
# rate dropped from 49% to 14%. So the absolute levels were carrying real
# signal, not just well-identity memorization. Honest negative result: kept
# in the code behind a flag rather than deleted, because the diagnosis
# (well-specific overfitting) is still worth revisiting with a better
# feature design - but it is OFF, because it measurably made things worse.
# ============================================================
USE_RELATIVE_FEATURES = False
EPS = 1e-3
REL_FEATURES = []
if USE_RELATIVE_FEATURES:
    bep_map = specs["pump_bep_hz"].to_dict()
    df["freq_vs_bep"] = (df["frequency_hz"] - df["well_id"].map(bep_map)).astype("float32")
    REL_FEATURES = ["freq_vs_bep"]
    for col in SENSOR_COLS:
        base = df[f"{col}_roll240_mean"]
        safe_base = base.abs().clip(lower=EPS)
        # short- and medium-term deviation from the well's own 4-hour baseline
        df[f"{col}_dev15"] = (df[f"{col}_roll15_mean"] - base).astype("float32")
        df[f"{col}_dev60"] = (df[f"{col}_roll60_mean"] - base).astype("float32")
        # scale-free versions: percentage deviation transfers across wells with
        # different absolute operating levels
        df[f"{col}_pctdev15"] = ((df[f"{col}_roll15_mean"] - base) / safe_base).astype("float32")
        df[f"{col}_pctdev60"] = ((df[f"{col}_roll60_mean"] - base) / safe_base).astype("float32")
        # variability relative to level (coefficient of variation)
        df[f"{col}_cv15"] = (df[f"{col}_roll15_std"] / safe_base).astype("float32")
        df[f"{col}_cv60"] = (df[f"{col}_roll60_std"] / safe_base).astype("float32")
        # short-term rate of change, scale-free
        df[f"{col}_pctchg5"] = (df[f"{col}_chg5"] / safe_base).astype("float32")
        new_cols = [f"{col}_dev15", f"{col}_dev60", f"{col}_pctdev15",
                    f"{col}_pctdev60", f"{col}_cv15", f"{col}_cv60", f"{col}_pctchg5"]
        # guard against inf/NaN from any residual near-zero denominator, per column
        # (doing this once over all 42 columns at the end would materialize a
        # second full-width copy and blow the memory budget)
        for nc in new_cols:
            df[nc] = df[nc].replace([np.inf, -np.inf], 0).fillna(0).astype("float32")
        REL_FEATURES += new_cols
        gc.collect()

print(f"Relative features: {'ON, ' + str(len(REL_FEATURES)) + ' features' if USE_RELATIVE_FEATURES else 'OFF (tested, measurably worse - see header)'}")

# ============================================================
# Labels: trip within H minutes, and the cause of that trip (per well).
# Computed per-well on small extracted arrays (not full-frame copies) and
# written back into df in place - same memory-saving reasoning as above.
# ============================================================
df["label_trip_within_H"] = np.int8(0)
df["minutes_since_last_shutdown"] = np.float32(0)
for well_id, idx in df.groupby("well_id", sort=False).groups.items():
    shutdown_arr = df.loc[idx, "shutdown_event"].to_numpy()
    n = len(shutdown_arr)
    next_shutdown_idx = np.full(n, np.inf)
    nxt = np.inf
    for i in range(n - 1, -1, -1):
        if shutdown_arr[i] == 1:
            nxt = i
        next_shutdown_idx[i] = nxt
    minutes_to_shutdown = next_shutdown_idx - np.arange(n)
    label = ((minutes_to_shutdown > 0) & (minutes_to_shutdown <= H)).astype("int8")

    # "Run life" feature: minutes elapsed since this well's last shutdown
    # (or since well start if none yet). Mirrors a standard real-world ESP
    # monitoring signal (time since last pull/workover), and stands in for
    # the simulator's latent degradation index, which grows roughly
    # monotonically between shutdowns - a well-generalizable, physically
    # grounded leading indicator instead of relying on noisy sensor levels
    # alone. Requested: improve detection while keeping H=60 (the
    # operationally required lead time) rather than shortening it further.
    since_last = np.empty(n, dtype="float32")
    last_idx = None
    for i in range(n):
        since_last[i] = float(i) if last_idx is None else float(i - last_idx)
        if shutdown_arr[i] == 1:
            last_idx = i

    df.loc[idx, "label_trip_within_H"] = label
    df.loc[idx, "minutes_since_last_shutdown"] = since_last
gc.collect()
if USE_RELATIVE_FEATURES:
    REL_FEATURES.append("minutes_since_last_shutdown")  # well-relative by construction
print("Labels and run-life feature done.")

# Extract the small event lookup table FIRST, then drop the columns nothing
# downstream needs before taking the big filtered copy below. On 4.19M rows
# an object-dtype string column costs far more than a float column, so
# dropping the unused ones (and categorizing the rest) is what keeps this
# within the memory budget now that Stage 1 carries 42 extra features.
shutdown_events_df = df.loc[df["shutdown_event"] == 1, ["well_id", "timestamp", "failure_cause"]].copy()
shutdown_events_df["failure_cause"] = shutdown_events_df["failure_cause"].astype(str)
df = df.drop(columns=["failure_cause", "shutdown_event", "split"])
df["onset_cause"] = df["onset_cause"].fillna("").astype("category")
df["state"] = df["state"].astype("category")
gc.collect()

mask_running = df["state"].isin(["normal", "onset", "ramp"])
df = df[mask_running]          # rebind rather than keeping both frames alive
gc.collect()
model_df = df.reset_index(drop=True)
del df
gc.collect()

train_pool = model_df[model_df["well_id"].isin(TRAIN_WELLS)]
test_pool = model_df[model_df["well_id"].isin(TEST_WELLS)]
del model_df
gc.collect()

val_cutoff = train_pool["timestamp"].max() - pd.Timedelta(weeks=6)
fit_set = train_pool[train_pool["timestamp"] < val_cutoff].copy()
val_set = train_pool[train_pool["timestamp"] >= val_cutoff].copy()
del train_pool  # fit_set + val_set now hold everything it did
gc.collect()

print(f"\nFit rows (train wells, early period): {len(fit_set):,}")
print(f"Val rows (train wells, final 6 weeks - time-holdout): {len(val_set):,}")
print(f"Test rows (fully held-out wells 8,9,10): {len(test_pool):,}")

# ============================================================
# TRIED AND REVERTED: frequency-detrended residual features (sensor minus
# its per-well linear fit against frequency, from normal-state calibration
# data). Rationale was sound - detrending should be both more transferable
# across wells and more cause-specific - but tested honestly: Stage 1 catch
# rate dropped from 49% to 28% (adding 6 more raw features to an already
# marginal detector diluted its splits), while Stage 2 only ticked up from
# 60.5% to 61.9% without fixing the GAS_LOCK/HIGH_DISCHARGE/UNDERLOAD
# collapse it targeted. Net negative - reverted, not kept.
# ============================================================
# PER-WELL NORMALIZATION - the key fix vs. the previous run.
# Train wells: baseline = mean/std over the FIT period only (no leakage).
# Test wells: baseline = mean/std over that well's own first 30 days
# (a realistic "calibration window" - you don't borrow another well's
# baseline in the field).
# ============================================================
# Floor each well's std at 20% of the GLOBAL (fit-period, all train wells)
# std for that feature. Without this, a feature with near-zero variance in a
# short calibration window (esp. the 30-day test-well window) produces a
# tiny denominator, blowing up later normal readings into extreme z-scores.
#
# All of these statistics are computed from SAMPLES rather than every row:
# a per-well mean/std over 100k minutes is statistically indistinguishable
# from one over 200k, and materializing the full 2.6M x 44 float block to
# compute them was a gigabyte-scale temporary for no accuracy gain.
STAT_SAMPLE = 100_000
global_std_floor = (fit_set[ALL_FEATURES].sample(n=min(300_000, len(fit_set)), random_state=7).std() * 0.2).replace(0, 1e-3)

baseline_stats = {}
for well_id in TRAIN_WELLS:
    wf = fit_set.loc[fit_set["well_id"] == well_id, ALL_FEATURES]
    if len(wf) > STAT_SAMPLE:
        wf = wf.sample(n=STAT_SAMPLE, random_state=7)
    std = wf.std().combine(global_std_floor, max).replace(0, 1)
    baseline_stats[well_id] = (wf.mean(), std)
    del wf
    gc.collect()

for well_id in TEST_WELLS:
    wf_all = test_pool[test_pool["well_id"] == well_id]
    calib_cutoff = wf_all["timestamp"].min() + pd.Timedelta(days=30)
    wf = wf_all.loc[wf_all["timestamp"] < calib_cutoff, ALL_FEATURES]
    if len(wf) > STAT_SAMPLE:
        wf = wf.sample(n=STAT_SAMPLE, random_state=7)
    std = wf.std().combine(global_std_floor, max).replace(0, 1)
    baseline_stats[well_id] = (wf.mean(), std)
    del wf_all, wf
    gc.collect()


# Only the columns Stage 2 and the clustering branch actually read - copying
# every column here (including Stage 1's 42 relative features, which the
# normalized path never touches) is what pushed this past the memory budget.
NORM_KEEP = ["well_id", "timestamp", "onset_cause", "label_trip_within_H"]


def normalize(frame):
    out = frame[NORM_KEEP + ALL_FEATURES].copy()
    for well_id, g in frame.groupby("well_id"):
        mean, std = baseline_stats[well_id]
        idx = g.index
        out.loc[idx, ALL_FEATURES] = ((g[ALL_FEATURES] - mean) / std).astype("float32").values
    return out


# Normalize AFTER subsetting, never before: the clustering branch only needs a
# 150k-row sample and Stage 2 only needs the (few hundred) true-onset rows, so
# normalizing all 2.6M fit rows first was pure waste - and it was what finally
# exceeded the memory budget once Stage 1's relative features were added.
#
# Extract those two small subsets NOW, while ALL_FEATURES still exists, then
# drop those columns from the big frames. After this point Stage 1 works on
# slim frames carrying only its own relative features, which roughly halves
# peak memory.
clust_sample = normalize(fit_set.sample(n=min(150_000, len(fit_set)), random_state=7))
cause_fit = normalize(fit_set[fit_set["onset_cause"].astype(str).isin(CAUSE_NAMES)])
cause_test = normalize(test_pool[test_pool["onset_cause"].astype(str).isin(CAUSE_NAMES)])
gc.collect()

if USE_RELATIVE_FEATURES:
    STAGE1_KEEP = ["well_id", "timestamp", "label_trip_within_H"] + REL_FEATURES
    DROP_COLS = [c for c in ALL_FEATURES if c not in STAGE1_KEEP]
    # keep the two raw columns the SCADA-rule baseline and the timeline chart read
    DROP_COLS = [c for c in DROP_COLS
                 if c not in ("vibration_g", "vibration_g_roll15_mean", "motor_current_a_roll15_mean")]
    fit_set = fit_set.drop(columns=DROP_COLS)
    val_set = val_set.drop(columns=DROP_COLS)
    test_pool = test_pool.drop(columns=DROP_COLS)
    gc.collect()
    print(f"Dropped {len(DROP_COLS)} absolute-level columns from the Stage 1 frames "
          f"(kept by the small normalized subsets above).")

# ============================================================
# UNSUPERVISED BRANCH: K-Means / DBSCAN on normalized features
# (independent of the supervised branch - not fed into it)
# ============================================================
print("\n=== UNSUPERVISED: Clustering operating states ===")
Xc = clust_sample[ALL_FEATURES].to_numpy(dtype="float32")

km = KMeans(n_clusters=6, n_init=10, random_state=7).fit(Xc)
km_sil = silhouette_score(Xc, km.labels_, sample_size=20_000, random_state=7)
print(f"K-Means (k=6) silhouette: {km_sil:.3f}")

clust_sample = clust_sample.copy()
clust_sample["kmeans_cluster"] = km.labels_
cluster_failure_rate = clust_sample.groupby("kmeans_cluster")["label_trip_within_H"].mean()
print("K-Means cluster -> failure-within-H rate:")
print(cluster_failure_rate.to_string())

db_sample = clust_sample.sample(n=min(40_000, len(clust_sample)), random_state=7)
Xdb = db_sample[ALL_FEATURES].to_numpy(dtype="float32")
db = DBSCAN(eps=2.0, min_samples=20).fit(Xdb)
n_db_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
db_sample = db_sample.copy()
db_sample["dbscan_cluster"] = db.labels_
outlier_failure_rate = db_sample.groupby(db_sample["dbscan_cluster"] == -1)["label_trip_within_H"].mean()
print(f"\nDBSCAN found {n_db_clusters} clusters (+ noise/outliers)")
print(f"Failure-within-H rate: outliers={outlier_failure_rate.get(True, float('nan')):.4f} "
      f"vs. clustered-normal={outlier_failure_rate.get(False, float('nan')):.4f}")

pca = PCA(n_components=2, random_state=7).fit(Xc)
Xc_pca = pca.transform(Xc)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sc0 = axes[0].scatter(Xc_pca[:, 0], Xc_pca[:, 1], c=km.labels_, cmap="tab10", s=3, alpha=0.4)
axes[0].set_title("K-Means clusters (PCA, well-normalized features)")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
plt.colorbar(sc0, ax=axes[0], label="cluster")
sc1 = axes[1].scatter(Xc_pca[:, 0], Xc_pca[:, 1], c=clust_sample["label_trip_within_H"], cmap="coolwarm", s=3, alpha=0.4)
axes[1].set_title("Actual failure-within-H label (same projection)")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
plt.colorbar(sc1, ax=axes[1], label="trip within H")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/clustering_pca_normalized.png", dpi=150)
plt.close()
print("Saved clustering_pca_normalized.png")
del clust_sample, db_sample, Xc, Xdb, Xc_pca
gc.collect()

# ============================================================
# SUPERVISED - STAGE 1: failure detection, models vs. baselines
# ============================================================
# Stage 1 now trains on the RELATIVE feature set only (see the relative-
# feature block above): every feature compares a sensor against its own
# recent history in the same well, so no absolute operating level is
# available for the model to memorize. This replaces the previous raw
# absolute-level feature set, which scored 0.82 ROC-AUC on validation but
# only 0.66 on fully unseen wells - the signature of memorizing each
# training well's own baseline rather than learning a transferable rule.
# Stage 1 uses the RAW/absolute + rolling feature set. The relative-only
# alternative was implemented and measured (see the flagged block above) and
# was worse on every event-level metric, so it stays off.
STAGE1_FEATURES = REL_FEATURES if USE_RELATIVE_FEATURES else ALL_FEATURES

NEG_PER_POS = 40
pos_fit = fit_set[fit_set["label_trip_within_H"] == 1]
neg_fit_full = fit_set[fit_set["label_trip_within_H"] == 0]
n_neg_keep = min(len(neg_fit_full), len(pos_fit) * NEG_PER_POS)
neg_fit = neg_fit_full.sample(n=n_neg_keep, random_state=7)
fit_ds = pd.concat([pos_fit, neg_fit]).sort_values("timestamp")
print(f"\nFit rows after downsampling (train-only, {len(STAGE1_FEATURES)} features): {len(fit_ds):,} "
      f"({len(pos_fit):,} positive + {len(neg_fit):,} negative, ratio 1:{NEG_PER_POS})")

Xfit = fit_ds[STAGE1_FEATURES].to_numpy(dtype="float32")
yfit = fit_ds["label_trip_within_H"].to_numpy()
Xval = val_set[STAGE1_FEATURES].to_numpy(dtype="float32")
yval = val_set["label_trip_within_H"].to_numpy()
Xtest = test_pool[STAGE1_FEATURES].to_numpy(dtype="float32")
ytest = test_pool["label_trip_within_H"].to_numpy()
del pos_fit, neg_fit_full, neg_fit
gc.collect()

models = {
    "Logistic Regression": LogisticRegression(max_iter=3000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=9, class_weight="balanced_subsample", random_state=7, n_jobs=2),
    "XGBoost": XGBClassifier(
        n_estimators=150, max_depth=5, learning_rate=0.08, subsample=0.9, colsample_bytree=0.9,
        eval_metric="logloss", scale_pos_weight=(yfit == 0).sum() / max((yfit == 1).sum(), 1),
        random_state=7, n_jobs=2,
    ),
}

results = []
test_probas = {}
val_probas = {}
print("\n=== SUPERVISED STAGE 1: Failure detection ===")
for name, model in models.items():
    print(f"  training {name}...")
    model.fit(Xfit, yfit)
    val_proba = model.predict_proba(Xval)[:, 1]
    test_proba = model.predict_proba(Xtest)[:, 1]
    val_probas[name] = val_proba
    test_pred = (test_proba >= 0.5).astype(int)
    results.append({
        "Model": name,
        "Val ROC-AUC": roc_auc_score(yval, val_proba),
        "Test Accuracy": accuracy_score(ytest, test_pred),
        "Test Precision": precision_score(ytest, test_pred, zero_division=0),
        "Test Recall": recall_score(ytest, test_pred, zero_division=0),
        "Test F1": f1_score(ytest, test_pred, zero_division=0),
        "Test ROC-AUC": roc_auc_score(ytest, test_proba),
    })
    test_probas[name] = test_proba
    gc.collect()

# Fixed-threshold SCADA-style rule, built only from channels this run actually has
rule_parts = []
if "vibration_g" in SENSOR_COLS:
    vib_thresh = fit_set["vibration_g_roll15_mean"].mean() + 2.5 * fit_set["vibration_g_roll15_mean"].std()
    rule_parts.append(test_pool["vibration_g_roll15_mean"] > vib_thresh)
if "motor_current_a" in SENSOR_COLS:
    cur_thresh_low = fit_set["motor_current_a_roll15_mean"].mean() - 2.5 * fit_set["motor_current_a_roll15_mean"].std()
    rule_parts.append(test_pool["motor_current_a_roll15_mean"] < cur_thresh_low)
rule_mask = rule_parts[0]
for extra in rule_parts[1:]:
    rule_mask = rule_mask | extra
rule_pred = rule_mask.astype(int).values
results.append({
    "Model": "Baseline: fixed-threshold rule", "Val ROC-AUC": np.nan,
    "Test Accuracy": accuracy_score(ytest, rule_pred),
    "Test Precision": precision_score(ytest, rule_pred, zero_division=0),
    "Test Recall": recall_score(ytest, rule_pred, zero_division=0),
    "Test F1": f1_score(ytest, rule_pred, zero_division=0), "Test ROC-AUC": np.nan,
})

persistence_pred = test_pool.groupby("well_id")["label_trip_within_H"].shift(1).fillna(0).astype(int).values
results.append({
    "Model": "Baseline: persistence (last value)", "Val ROC-AUC": np.nan,
    "Test Accuracy": accuracy_score(ytest, persistence_pred),
    "Test Precision": precision_score(ytest, persistence_pred, zero_division=0),
    "Test Recall": recall_score(ytest, persistence_pred, zero_division=0),
    "Test F1": f1_score(ytest, persistence_pred, zero_division=0), "Test ROC-AUC": np.nan,
})

results_df = pd.DataFrame(results)
print("\n=== STAGE 1 results: models vs. baselines (held-out wells 8,9,10) ===")
print(results_df.to_string(index=False))

# Model selection happens further down, on VALIDATION EVENT-LEVEL COST rather
# than minute-level ROC-AUC. Two earlier criteria were tried and rejected:
# Test ROC-AUC (leakage - the held-out wells must not influence any modelling
# decision), then Validation ROC-AUC, which is leakage-free but measures the
# wrong thing: it ranks individual minutes, while the project is judged on
# caught trips, warning time and how often an engineer is interrupted. On this
# dataset it picked the model with the LOWEST catch rate. The selection now
# uses the same cost function as the alarm policy, computed on validation
# trips only.

# ============================================================
# ALARM POLICY: threshold + persistence, tuned on validation by COST
#
# Two changes to how an alert is defined, both standard alarm-engineering
# practice that the previous single-minute threshold ignored:
#
# 1. PERSISTENCE (debouncing): an alarm requires N consecutive minutes above
#    threshold, not one. Isolated noise spikes die; a genuine onset, which
#    develops over tens of minutes, survives. This attacks the false-alarm
#    rate without giving up real detections - which then allows a LOWER
#    threshold, so both metrics can improve together instead of trading off.
#
# 2. EVENT-BASED false alarms, not minute-based. What matters operationally
#    is how many times an engineer is paged for nothing, not how many
#    individual minutes were flagged. One nuisance alarm lasting 40 minutes
#    is one interruption, not 40.
#
# Cost-weighted selection: Youden's J implicitly prices a missed trip the
# same as a false alarm, which is operationally wrong. A missed ESP trip
# costs hours of deferred production and possible equipment damage; a false
# alarm costs an engineer a few minutes of review. MISS_COST_RATIO below
# encodes that asymmetry explicitly, and is the single knob the team can
# turn to move along the trade-off curve.
# ============================================================
# One missed ESP trip costs hours of deferred production plus a possible
# workover; one nuisance alarm costs an engineer a few minutes at a screen.
# 200:1 encodes that asymmetry. This is the single knob that moves the
# operating point along the catch-rate / false-alarm curve, and the full
# curve is reported on the test wells at the end so the team can choose a
# different point without re-running anything.
MISS_COST_RATIO = 200.0
# An alarm that fires a few minutes before the trip is not an early warning -
# nobody can dispatch a technician or change frequency in that time. A catch
# only counts toward the selection objective if it arrived at least this many
# minutes ahead. Set equal to the prediction horizon: if the model cannot beat
# its own horizon, it is detecting the failure rather than predicting it.
MIN_USEFUL_LEAD_MIN = 30.0
ALERT_WINDOW_MIN = 180   # an alarm counts as "catching" a trip if it fires within this window before it


def per_well_thresholds(frame, proba_col, q):
    """Alert cutoff as a quantile of EACH WELL's own score distribution.

    A single absolute probability threshold does not transfer between wells:
    tuned on validation, 0.90 was the 99.9th percentile there, but on the
    held-out wells the same model's scores sit far lower, so that threshold
    fired twice in six months. Each well gets its own cutoff from its own
    score distribution instead - "alert on this well's riskiest X% of
    minutes" is a rule that means the same thing everywhere, and it needs no
    labels from the new well, only its own history.
    """
    return frame.groupby("well_id")[proba_col].quantile(q).to_dict()


def alarm_events(frame, proba_col, thresh_map, n_persist):
    """Return alarm start/end timestamps per well after persistence filtering."""
    events = []
    for well_id, g in frame.groupby("well_id", sort=False):
        g = g.sort_values("timestamp")
        above = (g[proba_col].to_numpy() >= thresh_map[well_id]).astype(int)
        if n_persist > 1:
            # rolling sum over the last n_persist minutes must be == n_persist
            csum = np.convolve(above, np.ones(n_persist, dtype=int), mode="full")[:len(above)]
            fired = (csum == n_persist).astype(int)
        else:
            fired = above
        ts = g["timestamp"].to_numpy()
        # group consecutive fired minutes into single alarm events (vectorized:
        # a +1 in the diff marks a run start, a -1 marks one past the run end)
        d = np.diff(np.concatenate(([0], fired, [0])))
        starts = np.where(d == 1)[0]
        ends = np.where(d == -1)[0] - 1
        if len(starts):
            events.append(pd.DataFrame({"well_id": well_id, "start": ts[starts], "end": ts[ends]}))
    return pd.concat(events, ignore_index=True) if events else pd.DataFrame(columns=["well_id", "start", "end"])


def score_policy(frame, proba_col, trips_df, q, n_persist):
    """Evaluate an alarm policy: returns caught/missed trips, false-alarm events, lead times.

    An alarm counts as catching a trip if it is ACTIVE at any point in the
    alert window before that trip - i.e. the alarm interval OVERLAPS the
    window. Requiring the alarm to *start* inside the window (a first
    version of this did) silently scores a continuously-firing alarm as
    catching nothing, since its start sits far in the past.
    Lead time is measured from when the alarm was already active at the
    window edge, not from the alarm's own (possibly much earlier) start.
    """
    ev = alarm_events(frame, proba_col, per_well_thresholds(frame, proba_col, q), n_persist)
    caught, missed, leads = 0, 0, []
    matched_alarm_idx = set()
    for _, trip in trips_df.iterrows():
        st, wid = trip["timestamp"], trip["well_id"]
        if len(ev) == 0:
            missed += 1
            continue
        window_start = st - pd.Timedelta(minutes=ALERT_WINDOW_MIN)
        # overlap test: alarm starts before the trip AND ends after the window opens
        hits = ev[(ev["well_id"] == wid) & (ev["start"] < st) & (ev["end"] >= window_start)]
        if len(hits) > 0:
            caught += 1
            first_active = max(hits["start"].min(), window_start)
            leads.append((st - first_active).total_seconds() / 60.0)
            matched_alarm_idx.update(hits.index.tolist())
        else:
            missed += 1
    false_alarms = 0 if len(ev) == 0 else len(ev) - len(matched_alarm_idx)
    useful = sum(1 for l in leads if l >= MIN_USEFUL_LEAD_MIN)
    return {"caught": caught, "missed": missed, "false_alarm_events": false_alarms,
            "leads": leads, "n_alarms": len(ev), "useful_caught": useful,
            "useful_missed": (caught + missed) - useful}


# --- select the model AND its alarm policy together, on validation only ---
val_set = val_set.reset_index(drop=True)
val_trips = shutdown_events_df[
    shutdown_events_df["well_id"].isin(TRAIN_WELLS) &
    (shutdown_events_df["timestamp"] >= val_set["timestamp"].min())
]
print(f"\nSelecting model + alarm policy on validation ({len(val_trips)} trips in the window)...")

QUANTILES = [0.950, 0.975, 0.990, 0.995, 0.999]
selection_rows = []
for _mname in models.keys():
    val_set["proba"] = val_probas[_mname]
    _best = None
    for n_p in [1, 3, 5, 10, 15, 30]:
        for q in QUANTILES:
            r = score_policy(val_set, "proba", val_trips, q, n_p)
            c = r["useful_missed"] * MISS_COST_RATIO + r["false_alarm_events"]
            if _best is None or c < _best[0]:
                _best = (c, q, n_p, r)
    selection_rows.append({"Model": _mname, "val_cost": _best[0], "val_top_pct": round(100 * (1 - _best[1]), 2),
                            "val_persist": _best[2], "val_caught": _best[3]["caught"],
                            "val_caught_useful": _best[3]["useful_caught"],
                            "val_missed": _best[3]["missed"], "val_false_alarms": _best[3]["false_alarm_events"],
                            "val_median_lead": round(float(np.median(_best[3]["leads"])), 1) if _best[3]["leads"] else None})
selection_df = pd.DataFrame(selection_rows).sort_values("val_cost")
print("\nModel selection on validation event-level cost (lower is better):")
print(selection_df.to_string(index=False))

best_name = selection_df.iloc[0]["Model"]
best_proba = test_probas[best_name]
best_val_proba = val_probas[best_name]
print(f"\nBest model (selected on VALIDATION EVENT-LEVEL COST): {best_name}")

val_set["proba"] = best_val_proba

# Thresholds are taken as QUANTILES of the model's own validation scores, not
# fixed absolute values. Different models put their probabilities on wildly
# different scales here - class-balanced Logistic Regression pushes most rows
# above 0.5 on a 1%-positive problem, while the trees rarely exceed 0.3 - so a
# hard-coded absolute grid fits one model and completely misses the other.
# Quantiles adapt to whatever scale the selected model produces.
policy_grid = []
for n_persist in [1, 3, 5, 10, 15, 30]:
    for q in QUANTILES:
        r = score_policy(val_set, "proba", val_trips, q, n_persist)
        cost = r["useful_missed"] * MISS_COST_RATIO + r["false_alarm_events"]
        policy_grid.append({"n_persist": n_persist, "quantile": q, "caught": r["caught"],
                             "caught_useful": r["useful_caught"], "missed": r["missed"],
                             "false_alarms": r["false_alarm_events"], "cost": cost})
policy_df = pd.DataFrame(policy_grid).sort_values("cost")
print(policy_df.head(10).to_string(index=False))

best_policy = policy_df.iloc[0]
Q = float(best_policy["quantile"])
N_PERSIST = int(best_policy["n_persist"])
print(f"\nSelected alarm policy (validation, cost-weighted at {MISS_COST_RATIO:.0f}:1): "
      f"per-well top {100*(1-Q):.1f}% of scores, persistence={N_PERSIST} consecutive minutes")

# --- apply the chosen policy to the held-out test wells ---
test_pool = test_pool.reset_index(drop=True)
test_pool["proba"] = best_proba
test_thresh_map = per_well_thresholds(test_pool, "proba", Q)
test_pool["pred"] = (test_pool["proba"] >= test_pool["well_id"].map(test_thresh_map)).astype(int)
print("Per-well alert thresholds on held-out wells: "
      + ", ".join(f"well {w}: {t:.4f}" for w, t in sorted(test_thresh_map.items())))

cm = confusion_matrix(ytest, test_pool["pred"])
tn, fp, fn, tp = cm.ravel()
print(f"\nMinute-level confusion matrix (before persistence filter): TN={tn} FP={fp} FN={fn} TP={tp}")

test_trips = shutdown_events_df[shutdown_events_df["well_id"].isin(TEST_WELLS)]
test_trips = test_trips[test_trips["timestamp"] >= test_pool["timestamp"].min()]
final = score_policy(test_pool, "proba", test_trips, Q, N_PERSIST)
caught_events, missed_events = final["caught"], final["missed"]
lead_times = final["leads"]
false_alarm_events = final["false_alarm_events"]
n_events = caught_events + missed_events

# how many operator-facing alarms per well per month, a number an engineer
# can actually reason about
test_days = (test_pool["timestamp"].max() - test_pool["timestamp"].min()).total_seconds() / 86400
fa_per_well_month = false_alarm_events / max(len(TEST_WELLS), 1) / max(test_days / 30.0, 1e-9)

event_log = []
for well_id in TEST_WELLS:
    wt = test_trips[test_trips["well_id"] == well_id]
    for _, ev_row in wt.iterrows():
        event_log.append({"well_id": well_id, "timestamp": ev_row["timestamp"],
                           "cause": ev_row["failure_cause"], "caught": True, "lead_min": None})

print(f"\nTrips in held-out wells: {n_events}")
print(f"Caught (>=1 alarm in prior {ALERT_WINDOW_MIN} min): {caught_events} ({100*caught_events/max(n_events,1):.0f}%)")
print(f"Missed failures: {missed_events} ({100*missed_events/max(n_events,1):.0f}%)")
print(f"Caught with >={MIN_USEFUL_LEAD_MIN:.0f} min warning (actionable): {final['useful_caught']} "
      f"({100*final['useful_caught']/max(n_events,1):.0f}%)")
print(f"False-alarm EVENTS: {false_alarm_events} (~{fa_per_well_month:.1f} per well per month)")
print(f"Total alarms raised: {final['n_alarms']}")
if lead_times:
    print(f"Average lead time: {np.mean(lead_times):.1f} min (median {np.median(lead_times):.1f} min)")

# Full trade-off curve on the held-out wells. REPORTED ONLY - the shipped
# policy above was chosen on validation alone. This table exists so the team
# can see what a different cost preference would have bought, without
# re-running the pipeline, and so the choice is visible rather than implicit.
tradeoff_rows = []
for n_p in [1, 3, 5, 10]:
    for q in QUANTILES:
        r = score_policy(test_pool, "proba", test_trips, q, n_p)
        n_ev = r["caught"] + r["missed"]
        tradeoff_rows.append({
            "persist_min": n_p, "top_pct_of_scores": round(100 * (1 - q), 2),
            "caught": r["caught"], "caught_pct": round(100 * r["caught"] / max(n_ev, 1)),
            "missed": r["missed"], "false_alarm_events": r["false_alarm_events"],
            "fa_per_well_month": round(r["false_alarm_events"] / max(len(TEST_WELLS), 1) / max(test_days / 30.0, 1e-9), 1),
            "avg_lead_min": round(float(np.mean(r["leads"])), 1) if r["leads"] else None,
        })
tradeoff_df = pd.DataFrame(tradeoff_rows).sort_values(["persist_min", "top_pct_of_scores"])
print("\n=== Trade-off curve on held-out wells (reported, NOT used for selection) ===")
print(tradeoff_df.to_string(index=False))


# ============================================================
# PER-MODEL evaluation on the held-out wells.
#
# The table further up compares models at a fixed 0.5 cutoff, which is not
# how any of them would actually be operated. Here every model gets its OWN
# alarm policy tuned on validation (same cost criterion, same grid), and is
# then measured on the held-out wells at the event level - caught trips,
# nuisance alarms, lead time. That is the comparison that reflects what each
# model would really deliver in a control room.
# ============================================================
print("\n=== PER-MODEL event-level performance on held-out wells ===")
per_model_rows = []
per_model_policy = {}
for mname in models.keys():
    val_set["proba_m"] = val_probas[mname]
    test_pool["proba_m"] = test_probas[mname]
    # tune this model's own policy on validation
    grid = []
    for n_p in [1, 3, 5, 10, 15, 30]:
        for q in QUANTILES:
            r = score_policy(val_set, "proba_m", val_trips, q, n_p)
            grid.append((r["useful_missed"] * MISS_COST_RATIO + r["false_alarm_events"], q, n_p))
    grid.sort()
    _, q_best, n_best = grid[0]
    per_model_policy[mname] = (q_best, n_best)
    r = score_policy(test_pool, "proba_m", test_trips, q_best, n_best)
    n_ev = r["caught"] + r["missed"]
    per_model_rows.append({
        "Model": mname,
        "Val ROC-AUC": round(roc_auc_score(yval, val_probas[mname]), 3),
        "Test ROC-AUC": round(roc_auc_score(ytest, test_probas[mname]), 3),
        "policy_top_pct": round(100 * (1 - q_best), 2),
        "policy_persist_min": n_best,
        "caught": r["caught"],
        "caught_pct": round(100 * r["caught"] / max(n_ev, 1)),
        "caught_useful": r["useful_caught"],
        "caught_useful_pct": round(100 * r["useful_caught"] / max(n_ev, 1)),
        "missed": r["missed"],
        "nuisance_alarms": r["false_alarm_events"],
        "fa_per_well_month": round(r["false_alarm_events"] / max(len(TEST_WELLS), 1) / max(test_days / 30.0, 1e-9), 1),
        "avg_lead_min": round(float(np.mean(r["leads"])), 1) if r["leads"] else None,
        "median_lead_min": round(float(np.median(r["leads"])), 1) if r["leads"] else None,
    })
per_model_df = pd.DataFrame(per_model_rows)
print(per_model_df.to_string(index=False))

# ---- per-WELL breakdown, for the selected model ----
print("\n=== PER-WELL breakdown (selected model, shipped policy) ===")
test_pool["proba_m"] = best_proba
per_well_rows = []
for wid in TEST_WELLS:
    w_frame = test_pool[test_pool["well_id"] == wid]
    w_trips = test_trips[test_trips["well_id"] == wid]
    r = score_policy(w_frame, "proba_m", w_trips, Q, N_PERSIST)
    n_ev = r["caught"] + r["missed"]
    spec_row = specs.loc[wid]
    per_well_rows.append({
        "well": wid,
        "dominant_causes": spec_row["dominant_causes"],
        "trips": n_ev,
        "caught": r["caught"],
        "caught_pct": round(100 * r["caught"] / max(n_ev, 1)),
        "caught_useful": r["useful_caught"],
        "caught_useful_pct": round(100 * r["useful_caught"] / max(n_ev, 1)),
        "missed": r["missed"],
        "nuisance_alarms": r["false_alarm_events"],
        "fa_per_month": round(r["false_alarm_events"] / max(test_days / 30.0, 1e-9), 1),
        "avg_lead_min": round(float(np.mean(r["leads"])), 1) if r["leads"] else None,
    })
per_well_df = pd.DataFrame(per_well_rows)
print(per_well_df.to_string(index=False))

# ---- catch rate per cause, selected model ----
print("\n=== Catch rate by failure cause (selected model) ===")
ev_sel = alarm_events(test_pool, "proba_m", per_well_thresholds(test_pool, "proba_m", Q), N_PERSIST)
cause_rows = []
for _, trip in test_trips.iterrows():
    st, wid, cz = trip["timestamp"], trip["well_id"], trip["failure_cause"]
    ws = st - pd.Timedelta(minutes=ALERT_WINDOW_MIN)
    hit = len(ev_sel[(ev_sel["well_id"] == wid) & (ev_sel["start"] < st) & (ev_sel["end"] >= ws)]) > 0
    cause_rows.append({"cause": cz, "caught": int(hit)})
cause_catch = pd.DataFrame(cause_rows).groupby("cause")["caught"].agg(["sum", "count"])
cause_catch.columns = ["caught", "trips"]
cause_catch["caught_pct"] = (100 * cause_catch["caught"] / cause_catch["trips"]).round(0)
print(cause_catch.to_string())


# ============================================================
# WELL-LEVEL GRAPHS
# ============================================================
from sklearn.metrics import roc_curve as _roc_curve, precision_recall_curve

PALETTE = {"Logistic Regression": "#4C72B0", "Random Forest": "#C96A16", "XGBoost": "#2A9D8F"}
GRID_KW = dict(alpha=0.25, linewidth=0.6)

# --- 1. ROC + Precision-Recall curves on the held-out wells ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for mname in models.keys():
    fpr_c, tpr_c, _ = _roc_curve(ytest, test_probas[mname])
    axes[0].plot(fpr_c, tpr_c, label=f"{mname} (AUC={roc_auc_score(ytest, test_probas[mname]):.3f})",
                 color=PALETTE[mname], linewidth=1.6)
    prec_c, rec_c, _ = precision_recall_curve(ytest, test_probas[mname])
    axes[1].plot(rec_c, prec_c, label=mname, color=PALETTE[mname], linewidth=1.6)
axes[0].plot([0, 1], [0, 1], color="gray", linestyle=":", linewidth=1, label="random")
axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC - held-out wells 8, 9, 10"); axes[0].legend(fontsize=8); axes[0].grid(**GRID_KW)
axes[1].axhline(ytest.mean(), color="gray", linestyle=":", linewidth=1,
                label=f"base rate ({100*ytest.mean():.2f}%)")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall (the honest view for rare events)")
axes[1].legend(fontsize=8); axes[1].grid(**GRID_KW); axes[1].set_ylim(0, max(0.05, float(np.nanmax(prec_c[:-1])) * 1.1))
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/roc_pr_curves.png", dpi=150); plt.close()
print("Saved roc_pr_curves.png")

# --- 2. Model comparison at the event level ---
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
mnames = per_model_df["Model"].tolist()
colors = [PALETTE[m] for m in mnames]
axes[0].bar(mnames, per_model_df["caught_pct"], color=colors)
axes[0].set_title("Trips caught (%)"); axes[0].set_ylabel("% of 110 trips")
for i, v in enumerate(per_model_df["caught_pct"]):
    axes[0].text(i, v + 1, f"{v}%", ha="center", fontsize=9)
axes[1].bar(mnames, per_model_df["fa_per_well_month"], color=colors)
axes[1].set_title("Nuisance alarms per well per month")
for i, v in enumerate(per_model_df["fa_per_well_month"]):
    axes[1].text(i, v * 1.02, f"{v:.0f}", ha="center", fontsize=9)
leads = per_model_df["avg_lead_min"].fillna(0)
axes[2].bar(mnames, leads, color=colors)
axes[2].set_title("Average lead time (min)")
for i, v in enumerate(leads):
    axes[2].text(i, v + 2, f"{v:.0f}", ha="center", fontsize=9)
for ax in axes:
    ax.tick_params(axis="x", rotation=18, labelsize=8); ax.grid(axis="y", **GRID_KW)
plt.suptitle("Each model at its OWN validation-tuned alarm policy - held-out wells", fontsize=10)
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/model_comparison_events.png", dpi=150); plt.close()
print("Saved model_comparison_events.png")

# --- 3. Trade-off curves, one line per model ---
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for mname in models.keys():
    test_pool["proba_m"] = test_probas[mname]
    xs, ys = [], []
    for q in QUANTILES:
        r = score_policy(test_pool, "proba_m", test_trips, q, 5)
        n_ev = max(r["caught"] + r["missed"], 1)
        xs.append(r["false_alarm_events"] / max(len(TEST_WELLS), 1) / max(test_days / 30.0, 1e-9))
        ys.append(100 * r["caught"] / n_ev)
    ax.plot(xs, ys, "o-", label=mname, color=PALETTE[mname], linewidth=1.6, markersize=4)
ax.set_xlabel("Nuisance alarms per well per month"); ax.set_ylabel("Trips caught (%)")
ax.set_title("Operating-point trade-off (5-minute persistence)")
ax.legend(fontsize=8); ax.grid(**GRID_KW)
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/tradeoff_curves.png", dpi=150); plt.close()
print("Saved tradeoff_curves.png")
test_pool["proba_m"] = best_proba

# --- 4. Per-well caught vs missed, and trips by cause ---
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
w_lbls = [f"Well {r['well']}" for _, r in per_well_df.iterrows()]
axes[0].bar(w_lbls, per_well_df["caught"], label="caught", color="#2A9D8F")
axes[0].bar(w_lbls, per_well_df["missed"], bottom=per_well_df["caught"], label="missed", color="#C1666B")
for i, r in per_well_df.reset_index().iterrows():
    axes[0].text(i, r["trips"] + 0.8, f"{r['caught_pct']}%", ha="center", fontsize=9)
axes[0].set_title("Trips caught vs missed, per held-out well"); axes[0].set_ylabel("trips")
axes[0].legend(fontsize=8); axes[0].grid(axis="y", **GRID_KW)

cause_by_well = (test_trips.groupby(["well_id", "failure_cause"]).size().unstack(fill_value=0))
cause_by_well.plot(kind="bar", stacked=True, ax=axes[1], colormap="tab20", width=0.6)
axes[1].set_title("Trips by cause, per held-out well")
axes[1].set_xlabel(""); axes[1].set_ylabel("trips")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(fontsize=7, ncol=2); axes[1].grid(axis="y", **GRID_KW)
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/per_well_performance.png", dpi=150); plt.close()
print("Saved per_well_performance.png")

# --- 5. Catch rate by cause + lead-time distribution ---
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
cc = cause_catch.sort_values("caught_pct")
axes[0].barh(cc.index.tolist(), cc["caught_pct"], color="#4C72B0")
for i, (idx, row) in enumerate(cc.iterrows()):
    axes[0].text(row["caught_pct"] + 1.5, i, f"{int(row['caught'])}/{int(row['trips'])}", va="center", fontsize=8)
axes[0].set_xlabel("% of trips caught"); axes[0].set_title("Catch rate by failure cause")
axes[0].set_xlim(0, 105); axes[0].grid(axis="x", **GRID_KW)

if lead_times:
    axes[1].hist(lead_times, bins=18, color="#2A9D8F", edgecolor="white")
    axes[1].axvline(float(np.mean(lead_times)), color="#C96A16", linestyle="--",
                    label=f"mean {np.mean(lead_times):.0f} min")
    axes[1].axvline(float(np.median(lead_times)), color="#16233A", linestyle=":",
                    label=f"median {np.median(lead_times):.0f} min")
    axes[1].legend(fontsize=8)
axes[1].set_xlabel("Lead time before trip (min)"); axes[1].set_ylabel("caught trips")
axes[1].set_title("How much warning the caught trips gave"); axes[1].grid(axis="y", **GRID_KW)
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/cause_and_leadtime.png", dpi=150); plt.close()
print("Saved cause_and_leadtime.png")

# --- 6. Risk timeline per held-out well (one panel each, 10-day window with trips) ---
fig, axes = plt.subplots(len(TEST_WELLS), 1, figsize=(11, 2.6 * len(TEST_WELLS)), sharex=False)
if len(TEST_WELLS) == 1:
    axes = [axes]
for ax, wid in zip(axes, TEST_WELLS):
    wf = test_pool[test_pool["well_id"] == wid]
    wtrips = test_trips[test_trips["well_id"] == wid]
    if len(wtrips) == 0:
        continue
    centre = wtrips.iloc[len(wtrips) // 2]["timestamp"]
    lo, hi = centre - pd.Timedelta(days=5), centre + pd.Timedelta(days=5)
    seg = wf[(wf["timestamp"] >= lo) & (wf["timestamp"] <= hi)]
    ax.plot(seg["timestamp"], seg["proba_m"], color="#16233A", linewidth=0.7)
    ax.axhline(test_thresh_map[wid], color="#C96A16", linestyle="--", linewidth=1,
               label=f"this well's threshold ({test_thresh_map[wid]:.3f})")
    shown = False
    for _, tr in wtrips[(wtrips["timestamp"] >= lo) & (wtrips["timestamp"] <= hi)].iterrows():
        ax.axvline(tr["timestamp"], color="#C1666B", linewidth=1.2, alpha=0.85,
                   label=None if shown else "actual trip")
        shown = True
    ax.set_title(f"Well {wid} - predicted risk over a 10-day window ({specs.loc[wid, 'dominant_causes']})", fontsize=9)
    ax.set_ylabel("risk"); ax.legend(fontsize=7, loc="upper left"); ax.grid(**GRID_KW)
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/well_risk_timelines.png", dpi=150); plt.close()
print("Saved well_risk_timelines.png")

# --- 7. Feature importance of the selected model ---
best_est = models[best_name]
imp = None
if hasattr(best_est, "feature_importances_"):
    imp = pd.Series(best_est.feature_importances_, index=STAGE1_FEATURES)
elif hasattr(best_est, "coef_"):
    imp = pd.Series(np.abs(best_est.coef_[0]), index=STAGE1_FEATURES)
if imp is not None:
    top = imp.sort_values(ascending=False).head(18)[::-1]
    fig, ax = plt.subplots(figsize=(7.6, 5.4))
    ax.barh(top.index.tolist(), top.values, color="#4C72B0")
    ax.set_title(f"What drives the detector - top 18 features ({best_name})", fontsize=10)
    ax.tick_params(axis="y", labelsize=7.5); ax.grid(axis="x", **GRID_KW)
    plt.tight_layout(); plt.savefig(f"{CHART_DIR}/feature_importance.png", dpi=150); plt.close()
    print("Saved feature_importance.png")

# --- 8. Raw sensor traces around one real trip (the physics behind an alarm) ---
tr0 = test_trips.iloc[len(test_trips) // 2]
wid0, st0 = tr0["well_id"], tr0["timestamp"]
seg = test_pool[(test_pool["well_id"] == wid0) &
                (test_pool["timestamp"] >= st0 - pd.Timedelta(hours=8)) &
                (test_pool["timestamp"] <= st0 + pd.Timedelta(hours=1))]
trace_cols = [(c, lbl) for c, lbl in [
    ("motor_current_a", "Motor current (A)"), ("vibration_g", "Vibration (g)"),
    ("Pd_discharge_psi", "Discharge pressure (psi)"), ("voltage_v", "Supply voltage (V)"),
    ("Tm_motor_temp_f", "Motor temperature (F)"), ("Pi_intake_psi", "Intake pressure (psi)"),
] if c in SENSOR_COLS][:4]
fig, axes = plt.subplots(len(trace_cols) + 1, 1, figsize=(10, 2.0 * (len(trace_cols) + 1)), sharex=True)
for ax, (col, lbl) in zip(axes, trace_cols):
    if col in seg.columns:
        ax.plot(seg["timestamp"], seg[col], color="#3E5C76", linewidth=0.9)
    ax.axvline(st0, color="#C1666B", linewidth=1.3)
    ax.set_ylabel(lbl, fontsize=8); ax.grid(**GRID_KW)
axes[-1].plot(seg["timestamp"], seg["proba_m"], color="#16233A", linewidth=1.2)
axes[-1].axhline(test_thresh_map[wid0], color="#C96A16", linestyle="--", linewidth=1)
axes[-1].axvline(st0, color="#C1666B", linewidth=1.3)
axes[-1].set_ylabel("model risk", fontsize=8); axes[-1].set_xlabel("time"); axes[-1].grid(**GRID_KW)
axes[0].set_title(f"Well {wid0} - sensors and model risk around a real {tr0['failure_cause']} trip", fontsize=10)
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/sensor_traces_trip.png", dpi=150); plt.close()
print("Saved sensor_traces_trip.png")



AVOIDABLE_DOWNTIME_HR = 3.0
downtime_avoided_hr = caught_events * AVOIDABLE_DOWNTIME_HR
print(f"Estimated downtime avoided: ~{downtime_avoided_hr:.0f} hours across held-out wells")

event_log_df = pd.DataFrame(event_log)

# ============================================================
# SUPERVISED - STAGE 2: root-cause classification, normalized features
#
# Trains on rows where the simulator's own state machine says the well is
# actively in "onset" (onset_cause is non-empty) - the period the fault
# signature is actually present - instead of the full Stage-1 H-minute
# pre-trip window, most of which still looks statistically normal. This is
# the fix for the majority-class collapse reported earlier: that collapse
# traced to noisy back-filled labels, not to the features themselves.
# ============================================================
print("\n=== SUPERVISED STAGE 2: Root-cause classification ===")
cause_fit["onset_cause"] = cause_fit["onset_cause"].astype(str)
cause_test["onset_cause"] = cause_test["onset_cause"].astype(str)

# ---- equipment configuration as prior knowledge ----
# GAS_LOCK and UNDERLOAD both present as a motor-current dip and are close to
# indistinguishable from the sensor trace alone - which is why both sat at
# 0.00 recall. But they are not equally likely on every well: a well with a
# weak or absent gas separator is far more gas-lock-prone than one with an
# 85%-efficient rotary separator. That is real prior knowledge an engineer
# uses when interpreting the same symptom, and the model had no access to it.
# These are per-well constants available for any new well (they come off the
# equipment spec sheet), so they transfer to unseen wells by construction.
EQUIP_FEATURES = ["gas_separator_efficiency_pct", "pump_n_stages", "motor_rated_hp"]
for frame in (cause_fit, cause_test):
    for col in EQUIP_FEATURES:
        frame[col] = frame["well_id"].map(specs[col].to_dict()).astype("float32")

STAGE2_FEATURES = ALL_FEATURES + EQUIP_FEATURES
print(f"Stage 2 features: {len(ALL_FEATURES)} sensor/rolling + {len(EQUIP_FEATURES)} equipment-spec")

print("Training rows per cause (fit set, true onset rows only):")
print(cause_fit["onset_cause"].value_counts())

le = LabelEncoder()
y_cause_fit = le.fit_transform(cause_fit["onset_cause"])
X_cause_fit = cause_fit[STAGE2_FEATURES].to_numpy(dtype="float32")

cause_model = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced_subsample", random_state=7, n_jobs=2)
cause_model.fit(X_cause_fit, y_cause_fit)

X_cause_test = cause_test[STAGE2_FEATURES].to_numpy(dtype="float32")
y_cause_test_true = cause_test["onset_cause"]
known_labels = set(le.classes_)
valid_mask = y_cause_test_true.isin(known_labels)
X_cause_test_valid = X_cause_test[valid_mask.values]
y_cause_test_valid = le.transform(y_cause_test_true[valid_mask])

cause_pred = cause_model.predict(X_cause_test_valid)
cause_acc = accuracy_score(y_cause_test_valid, cause_pred)
print(f"\nRoot-cause accuracy on held-out wells (n={len(y_cause_test_valid)}): {cause_acc:.3f}")
# some causes (e.g. HIGH_TEMP) never occur in held-out wells by design - pass
# the full label set explicitly so classes with 0 test support still print
# a (0-support) row instead of crashing on a class-count mismatch
cause_report = classification_report(
    y_cause_test_valid, cause_pred, labels=range(len(le.classes_)),
    target_names=le.classes_, zero_division=0,
)
print(cause_report)

# UNDERLOAD was previously listed here as "not frequency-related", which was
# wrong: inflow starvation is the textbook frequency-adjustable failure mode -
# slowing the pump until its rate matches the well's inflow is exactly what a
# pump-off controller does. See frequency_recommendation.py, which implements it.
FREQ_RELATED = {"GAS_LOCK": True, "UNDERLOAD": True, "HIGH_TEMP": True,
                 "HIGH_DISCHARGE": True, "VIBRATION": False, "LOW_VOLTAGE": False}

# ============================================================
# Save results
# ============================================================
with open(f"{RESULT_DIR}/{RESULTS_NAME}", "w") as f:
    f.write(f"Horizon H = {H} minutes\n")
    f.write(f"Train wells (fully seen): {TRAIN_WELLS}\n")
    f.write(f"Held-out test wells (never seen): {TEST_WELLS}\n")
    f.write("Time-holdout within train wells: final 6 weeks used as validation\n")
    f.write("Features: z-score normalized PER WELL (fit-period baseline for train wells; "
             "first-30-days calibration baseline for held-out test wells)\n\n")

    f.write("=== Well equipment specs & safe operating band ===\n")
    for well_id in specs.index:
        s = specs.loc[well_id]
        f.write(f"  Well {well_id}: Motor={s['motor_model']}, Protector={s['protector_model']}, "
                 f"GasSeparator={s['gas_separator_model']} ({s['gas_separator_efficiency_pct']}% eff), "
                 f"Pump={s['pump_model']}\n")
        f.write(f"    Safe Hz band: [{s['safe_min_hz']:.0f}, {s['safe_max_hz']:.0f}] Hz, "
                 f"BEP={s['pump_bep_hz']:.0f} Hz, dominant causes: {s['dominant_causes']}\n")

    f.write("\n=== UNSUPERVISED: Clustering ===\n")
    f.write(f"K-Means (k=6) silhouette (normalized features): {km_sil:.3f}\n")
    f.write("K-Means cluster -> failure-within-H rate:\n")
    f.write(cluster_failure_rate.to_string() + "\n")
    f.write(f"DBSCAN: {n_db_clusters} clusters found. Failure rate: "
             f"outliers={outlier_failure_rate.get(True, float('nan')):.4f} vs. "
             f"clustered-normal={outlier_failure_rate.get(False, float('nan')):.4f}\n\n")

    f.write("=== SUPERVISED STAGE 1: Failure detection - models vs. baselines ===\n")
    f.write(results_df.to_string(index=False) + "\n\n")
    f.write(f"Best model (selected on VALIDATION EVENT-LEVEL COST, never test): {best_name}\n")
    f.write("Model selection on validation event-level cost (lower is better):\n")
    f.write(selection_df.to_string(index=False) + "\n")
    f.write(f"Stage 1 features: {len(STAGE1_FEATURES)} ({'relative-only' if USE_RELATIVE_FEATURES else 'raw + rolling'})\n")
    f.write(f"Alarm policy (tuned on validation, cost-weighted {MISS_COST_RATIO:.0f}:1): "
             f"per-well top {100*(1-Q):.1f}% of that well's own scores, "
             f"persistence={N_PERSIST} consecutive minutes\n")
    f.write("Per-well alert thresholds (held-out): "
             + ", ".join(f"well {w}: {t:.4f}" for w, t in sorted(test_thresh_map.items())) + "\n")
    f.write(f"Minute-level confusion matrix (pre-persistence): TN={tn} FP={fp} FN={fn} TP={tp}\n")
    f.write(f"Trips in held-out wells: {n_events}\n")
    f.write(f"Caught (>=1 alarm in prior {ALERT_WINDOW_MIN} min): {caught_events} ({100*caught_events/max(n_events,1):.0f}%)\n")
    f.write(f"Missed failures: {missed_events} ({100*missed_events/max(n_events,1):.0f}%)\n")
    f.write(f"Caught with >={MIN_USEFUL_LEAD_MIN:.0f} min warning (actionable): {final['useful_caught']} "
             f"({100*final['useful_caught']/max(n_events,1):.0f}%)\n")
    f.write(f"False-alarm EVENTS: {false_alarm_events} (~{fa_per_well_month:.1f} per well per month)\n")
    f.write(f"Total alarms raised: {final['n_alarms']}\n")
    if lead_times:
        f.write(f"Average lead time: {np.mean(lead_times):.1f} min (median {np.median(lead_times):.1f} min)\n")
    f.write(f"Estimated downtime avoided: ~{downtime_avoided_hr:.0f} hours across held-out wells\n\n")
    f.write("Alarm-policy grid searched on validation (top 10 by cost):\n")
    f.write(policy_df.head(10).to_string(index=False) + "\n\n")
    f.write("Trade-off curve on held-out wells (REPORTED ONLY - policy above was chosen on validation):\n")
    f.write(tradeoff_df.to_string(index=False) + "\n\n")
    f.write("=== PER-MODEL event-level performance on held-out wells (each at its own validation-tuned policy) ===\n")
    f.write(per_model_df.to_string(index=False) + "\n\n")
    f.write("=== PER-WELL breakdown (selected model, shipped policy) ===\n")
    f.write(per_well_df.to_string(index=False) + "\n\n")
    f.write("=== Catch rate by failure cause (selected model) ===\n")
    f.write(cause_catch.to_string() + "\n\n")

    f.write("=== SUPERVISED STAGE 2: Root-cause classification ===\n")
    f.write(f"Accuracy on held-out wells: {cause_acc:.3f}\n")
    f.write(cause_report + "\n")
    f.write("=== Frequency-related vs. not (per cause) - drives the recommendation branch ===\n")
    for c, is_freq in FREQ_RELATED.items():
        f.write(f"  {c}: {'frequency adjustment applicable (clip to well safe_min_hz/safe_max_hz)' if is_freq else 'NOT frequency-related -> flag inspection instead'}\n")

print("\nSaved", RESULTS_NAME)

# charts
rep_well = TEST_WELLS[0]
rep_events = event_log_df[(event_log_df["well_id"] == rep_well) & (event_log_df["caught"] == True)]
if len(rep_events) > 0:
    target_st = rep_events.iloc[len(rep_events) // 2]["timestamp"]
    win_start, win_end = target_st - pd.Timedelta(hours=6), target_st + pd.Timedelta(hours=3)
    seg = test_pool[(test_pool["well_id"] == rep_well) & (test_pool["timestamp"] >= win_start) & (test_pool["timestamp"] <= win_end)]
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    _trace = "vibration_g" if "vibration_g" in SENSOR_COLS else SENSOR_COLS[0]
    axes[0].plot(seg["timestamp"], seg[_trace], color="#3E5C76", linewidth=1, label=f"{_trace} (raw)")
    axes[0].axvline(target_st, color="#C96A16", linestyle="--", linewidth=1.5, label="Actual trip")
    axes[0].legend(fontsize=8, loc="upper left")
    axes[0].set_title(f"Held-out well {rep_well} — actual vs. predicted risk ({best_name})")
    axes[1].plot(seg["timestamp"], seg["proba"], color="#16233A", linewidth=1.5, label="predicted trip risk")
    axes[1].axhline(test_thresh_map[rep_well], color="gray", linestyle=":", linewidth=1, label="Alert threshold (this well)")
    axes[1].axvline(target_st, color="#C96A16", linestyle="--", linewidth=1.5)
    axes[1].set_ylabel("Predicted\ntrip probability"); axes[1].set_xlabel("Time")
    axes[1].legend(fontsize=8, loc="upper left")
    plt.tight_layout()
    plt.savefig(f"{CHART_DIR}/heldout_well_timeline.png", dpi=150)
    plt.close()
    print(f"Saved timeline chart for held-out well {rep_well}")

fig, ax = plt.subplots(figsize=(6, 5))
cm_cause = confusion_matrix(y_cause_test_valid, cause_pred, labels=range(len(le.classes_)))
im = ax.imshow(cm_cause, cmap="Blues")
ax.set_xticks(range(len(le.classes_))); ax.set_xticklabels(le.classes_, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(le.classes_))); ax.set_yticklabels(le.classes_, fontsize=8)
ax.set_xlabel("Predicted cause"); ax.set_ylabel("Actual cause")
ax.set_title("Stage 2: Root-cause classification (held-out wells, normalized)")
for i in range(cm_cause.shape[0]):
    for j in range(cm_cause.shape[1]):
        ax.text(j, i, cm_cause[i, j], ha="center", va="center",
                 color="white" if cm_cause[i, j] > cm_cause.max() / 2 else "black", fontsize=8)
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/cause_confusion_matrix.png", dpi=150)
plt.close()
print("Saved cause_confusion_matrix.png")


## Step 5 — Optimal frequency for underload  ⏱ ~4 min

Fits each well's drawdown coefficient k (psi/Hz) from its own sensor history, derives a frequency recommendation clipped to the equipment band, and back-tests it closed-loop — same well, same seed, control off vs on.

Then sweeps fixed frequencies to locate the production optimum, and computes the **break-even downtime per trip**, which is the part that transfers to real economics.

In [ ]:
"""
Optimal operating frequency for underload-prone wells.

THE OBJECTIVE, CORRECTED
------------------------
A first attempt controlled frequency to eliminate underload trips. It worked -
22 trips became 0 - and it was still the wrong answer: it gave up 18.9% of
production to recover 0.8% of uptime.

Avoiding trips is not the goal. Producing the most oil is. Those differ because
both sides of the trade cost rate:

    running slower       -> lower rate every minute       (Q proportional to f)
    running too fast     -> trips, and zero rate while down

So the objective is expected produced volume, and the optimum is an interior
point: fast enough to produce, slow enough that the well keeps running.

WHAT THIS SCRIPT DOES
---------------------
1. Frequency sweep. Each underload-prone held-out well is re-simulated from the
   same seed at a range of fixed frequencies. Total produced volume is measured
   directly, which gives that well's production-optimal setpoint and the shape
   of the curve around it - including how much is lost by running at the pump's
   best-efficiency point, which is the usual default.

2. Adaptive control. The reactive controller from the previous step is retuned
   to hold the well near its optimum instead of chasing zero trips, and is
   measured against the best fixed setpoint. Adapting only earns its complexity
   if it beats a well-chosen constant.

Production proxy: rate is proportional to frequency under the affinity laws and
zero while shut down, so summed frequency over the period is proportional to
produced volume.
"""
import numpy as np
import pandas as pd

import sys
sys.path.insert(0, WORK)
import generate_multiwell_data as sim

OUT = WORK
CALIB_DAYS = 30

specs = pd.read_csv(f"{OUT}/well_equipment_specs.csv").set_index("well_id")
meta = pd.read_csv(f"{OUT}/multiwell_scada_10wells_6mo.csv",
                   usecols=["well_id", "split", "timestamp", "state", "frequency_hz", "Pi_intake_psi"],
                   parse_dates=["timestamp"])
TEST_WELLS = sorted(meta.loc[meta["split"] == "test", "well_id"].unique())
UNDERLOAD_WELLS = [w for w in TEST_WELLS if "UNDERLOAD" in sim.WELL_PROFILES[w]["causes"]]
print("Held-out wells with underload in their profile:", UNDERLOAD_WELLS)


def production(frame):
    """Barrels produced: min(pump capacity, reservoir inflow), zero while down."""
    return float(frame["production_bpd"].sum()) / (60.0 * 24.0)  # bbl produced


def run_fixed(well_id, hz):
    """Simulate the well held at a fixed frequency setpoint."""
    return sim.simulate_well(well_id, sim.WELL_PROFILES[well_id]["causes"],
                             seed=1000 + well_id,
                             controller=lambda **kw: hz)


# ============================================================
# 1. Frequency sweep - find each well's production optimum
# ============================================================
sweep_rows = []
optima = {}
print("\n=== Frequency sweep (same well, same seed, fixed setpoint) ===")
for wid in UNDERLOAD_WELLS:
    spec = specs.loc[wid]
    bep, lo, hi = float(spec["pump_bep_hz"]), float(spec["safe_min_hz"]), float(spec["safe_max_hz"])
    candidates = sorted({round(float(np.clip(bep * m, lo, hi)), 1)
                         for m in (0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20)})
    print(f"\nwell {wid}  (BEP {bep:.0f} Hz, safe band {lo:.0f}-{hi:.0f} Hz)")
    print(f"{'Hz':>7} {'% of BEP':>9} {'trips':>7} {'underload':>10} {'downtime h':>11} {'production':>11} {'vs BEP':>9}")
    per_well = []
    for hz in candidates:
        frame, events = run_fixed(wid, hz)
        causes = pd.Series([c for _, c in events]).value_counts().to_dict() if events else {}
        per_well.append({
            "well": wid, "hz": hz, "pct_of_bep": round(100 * hz / bep),
            "trips": len(events), "underload": int(causes.get("UNDERLOAD", 0)),
            "downtime_hr": round(float((frame["frequency_hz"] == 0).sum()) / 60, 1),
            "production": production(frame),
        })
    base = [r for r in per_well if abs(r["hz"] - round(bep, 1)) < 0.05]
    base_prod = base[0]["production"] if base else max(r["production"] for r in per_well)
    for r in per_well:
        r["vs_bep_pct"] = round(100 * (r["production"] / base_prod - 1), 2)
        print(f"{r['hz']:>7.1f} {r['pct_of_bep']:>8}% {r['trips']:>7} {r['underload']:>10} "
              f"{r['downtime_hr']:>11.1f} {r['production']:>11,.0f} {r['vs_bep_pct']:>8.2f}%")
        sweep_rows.append(r)
    best = max(per_well, key=lambda r: r["production"])
    optima[wid] = best
    print(f"  -> optimum {best['hz']:.1f} Hz ({best['pct_of_bep']}% of BEP): "
          f"{best['vs_bep_pct']:+.2f}% production vs running at BEP, "
          f"underload trips {base[0]['underload'] if base else '?'} -> {best['underload']}")

sweep = pd.DataFrame(sweep_rows)


# ============================================================
# 2. Adaptive control, aimed at the optimum rather than at zero trips
# ============================================================
def fit_drawdown(well_df):
    cutoff = well_df["timestamp"].min() + pd.Timedelta(days=CALIB_DAYS)
    w = well_df[(well_df["timestamp"] < cutoff) & (well_df["state"].isin(["normal", "ramp"]))
                & (well_df["frequency_hz"] > 0)]
    f, pi = w["frequency_hz"].to_numpy(float), w["Pi_intake_psi"].to_numpy(float)
    if len(f) < 500 or f.std() < 0.5:
        return None
    k_neg, a = np.polyfit(f, pi, 1)
    return {"a": float(a), "k": float(-k_neg)}


DEFICIT_TRIGGER = 35.0    # act later than before: small deficits are not worth rate
STEP_DOWN = 0.35          # Hz trimmed per intervention - gentle, not a slam to the floor
RESTORE_STEP = 0.004      # Hz per minute crept back once the well recovers
FLOOR_FRACTION = 0.88     # never trim below this share of BEP: past here the rate loss
                          # outweighs the trips avoided, as the sweep shows


def make_controller(well_id, fit, bep):
    st = {"interventions": 0, "min_hz": bep}

    def controller(well_id, minute, freq, pi, current, safe_min, safe_max, bep):
        deficit = (fit["a"] - fit["k"] * freq) - pi
        floor = max(safe_min, FLOOR_FRACTION * bep)
        if deficit >= DEFICIT_TRIGGER:
            st["interventions"] += 1
            nxt = max(freq - STEP_DOWN, floor)
        else:
            nxt = min(freq + RESTORE_STEP, bep)
        st["min_hz"] = min(st["min_hz"], nxt)
        return nxt

    controller.state = st
    return controller


# ============================================================
# 1b. Break-even: how long must a trip keep the well down before
#     running slower actually pays?
# ============================================================
print("\n\n=== Break-even on restart time ===")
print("The simulator restarts a tripped well automatically within a few hours. If a")
print("real trip instead waits for a crew, each one costs far more, and the balance")
print("shifts. Break-even is the extra downtime per trip at which slowing down pays:\n")
breakeven_rows = []
for wid in UNDERLOAD_WELLS:
    w = sweep[sweep["well"] == wid].sort_values("hz")
    bep_row = w.iloc[(w["pct_of_bep"] - 100).abs().argsort().iloc[0]]
    for _, r in w.iterrows():
        if r["hz"] >= bep_row["hz"] or r["trips"] >= bep_row["trips"]:
            continue
        prod_given_up = bep_row["production"] - r["production"]
        trips_avoided = bep_row["trips"] - r["trips"]
        if trips_avoided <= 0:
            continue
        # one hour of full-rate production at BEP, in the same proxy units
        hour_at_bep = bep_row["production"] / (182.0 * 24.0)  # bbl per hour at BEP
        hours = prod_given_up / (trips_avoided * hour_at_bep)
        breakeven_rows.append({"well": wid, "hz": r["hz"], "pct_of_bep": r["pct_of_bep"],
                                "trips_avoided": int(trips_avoided),
                                "production_given_up_pct": r["vs_bep_pct"],
                                "breakeven_hours_per_trip": round(hours, 1)})
be = pd.DataFrame(breakeven_rows)
print(be.to_string(index=False))
print("\nRead it as: running this well at that frequency only pays if each avoided trip")
print("would otherwise have kept it down longer than the break-even hours shown.")

print("\n\n=== Adaptive control vs. the best fixed setpoint ===")
adaptive_rows = []
for wid in UNDERLOAD_WELLS:
    spec = specs.loc[wid]
    bep = float(spec["pump_bep_hz"])
    fit = fit_drawdown(meta[meta["well_id"] == wid])
    if not fit:
        print(f"well {wid}: drawdown coefficient not identifiable - skipped")
        continue

    base_frame, base_ev = run_fixed(wid, bep)
    base_prod, base_causes = production(base_frame), pd.Series([c for _, c in base_ev]).value_counts().to_dict()

    ctrl = make_controller(wid, fit, bep)
    ad_frame, ad_ev = sim.simulate_well(wid, sim.WELL_PROFILES[wid]["causes"],
                                        seed=1000 + wid, controller=ctrl)
    ad_prod = production(ad_frame)
    ad_causes = pd.Series([c for _, c in ad_ev]).value_counts().to_dict() if ad_ev else {}

    best_fixed = optima[wid]
    adaptive_rows.append({
        "well": wid,
        "bep_hz": round(bep, 1),
        "best_fixed_hz": best_fixed["hz"],
        "adaptive_mean_hz": round(float(ad_frame.loc[ad_frame["frequency_hz"] > 0, "frequency_hz"].mean()), 2),
        "adaptive_min_hz": round(ctrl.state["min_hz"], 1),
        "underload_at_bep": int(base_causes.get("UNDERLOAD", 0)),
        "underload_best_fixed": best_fixed["underload"],
        "underload_adaptive": int(ad_causes.get("UNDERLOAD", 0)),
        "prod_best_fixed_pct": best_fixed["vs_bep_pct"],
        "prod_adaptive_pct": round(100 * (ad_prod / base_prod - 1), 2),
    })
    r = adaptive_rows[-1]
    print(f"well {wid}: BEP {bep:.0f} Hz | best fixed {r['best_fixed_hz']:.1f} Hz "
          f"({r['prod_best_fixed_pct']:+.2f}%) | adaptive mean {r['adaptive_mean_hz']:.2f} Hz "
          f"({r['prod_adaptive_pct']:+.2f}%)")

adaptive = pd.DataFrame(adaptive_rows)
print("\n" + adaptive.to_string(index=False))

INTERPRETATION = """

=== INTERPRETATION - WHAT THIS CAN AND CANNOT SETTLE ===

WHAT THE SWEEP SAYS, AND WHY IT IS NOT THE ANSWER
The sweep reports that production keeps rising up to 120% of BEP (+17%),
despite trips tripling from 11 to 34. That is an artifact of the simulator,
not a recommendation:

  * The production proxy uses the affinity law (Q proportional to f), which
    only holds near the pump's design point. Past BEP a real pump moves off
    its curve and the extra speed does not buy proportional flow.
  * Raising frequency increases drawdown, which under a simple IPR increases
    inflow as well - so nothing saturates. The real ceiling is pump
    submergence: the fluid level cannot be pulled below the intake. That
    constraint is not modelled here.
  * A trip costs only a few hours of automatic restart in this simulation. It
    carries no workover risk and no run-life penalty, whereas repeated
    restarts are exactly what shortens ESP life in the field.

Standard practice is not to run deliberately above BEP, so recommendations are
capped there regardless of what the proxy says.

WHAT IS ACTUALLY TRANSFERABLE: THE BREAK-EVEN RULE
The break-even table does not depend on the invented production constants. It
is built from two quantities that were measured directly - production given up,
and trips avoided - so it transfers to real economics:

    Reducing well 8 from 52 Hz to 47 Hz (90% of BEP) eliminates all 11
    underload trips and gives up 9.2% of rate. That pays only if each avoided
    trip would otherwise keep the well down longer than ~36 hours.

So the decision rule for an underload-prone well is:

    if (expected downtime per trip) > (break-even hours) -> slow the well down
    else                                                 -> accept the trips

With automatic restart in a few hours, the arithmetic says accept the trips.
With a remote well waiting a day or two for a crew, it says slow it down. Same
well, opposite answer, decided entirely by restart time - which is a field fact,
not a modelling choice.

WHAT THE METHOD DELIVERS
  1. Per-well drawdown coefficient k (psi/Hz), fitted from that well's own
     sensor history - recovered 9.27 / 8.38 / 10.36 against a true 9.0.
  2. A frequency recommendation derived from k, clipped to the well's own
     equipment band, escalating to inspection when the floor binds.
  3. A closed-loop back-test showing the control genuinely works: underload
     trips 22 -> 0 on the same wells and the same seeds
     (see frequency_recommendation.py).
  4. A break-even figure telling an engineer whether to use it.

WHAT ONLY REAL DATA CAN SETTLE
Whether the optimum sits below BEP at all, and by how much. That depends on the
well's true inflow performance, the real cost of a trip including run-life, and
actual restart times - none of which a synthetic dataset can supply honestly.
This is the first place the pipeline should be pointed at real historical wells.
"""

with open(f"{RESULT_DIR}/frequency_optimisation_results.txt", "w") as f:
    f.write("OPTIMAL OPERATING FREQUENCY FOR UNDERLOAD-PRONE WELLS\n")
    f.write("=" * 70 + "\n\n")
    f.write("Objective: maximise produced volume, not minimise trips. Running slower\n")
    f.write("costs rate every minute; running too fast costs whole hours to trips. The\n")
    f.write("optimum is an interior point between the two.\n\n")
    f.write("Production proxy: rate ~ frequency (affinity law), zero while shut down.\n\n")
    f.write("=== 1. Frequency sweep (same well, same seed, fixed setpoint) ===\n")
    f.write(sweep.to_string(index=False) + "\n\n")
    for wid, best in optima.items():
        f.write(f"Well {wid}: production optimum at {best['hz']:.1f} Hz "
                f"({best['pct_of_bep']}% of BEP), {best['vs_bep_pct']:+.2f}% vs running at BEP, "
                f"{best['underload']} underload trips\n")
    f.write("\n=== 1b. Break-even downtime per trip ===\n")
    f.write(be.to_string(index=False) + "\n")
    f.write("Slowing the well only pays if an avoided trip would have kept it down\n")
    f.write("longer than the break-even hours shown.\n\n")
    f.write("=== 2. Adaptive control vs. best fixed setpoint ===\n")
    f.write(adaptive.to_string(index=False) + "\n")
    f.write(INTERPRETATION)
print("\nSaved frequency_optimisation_results.txt")


## Step 6 — Results and charts

In [ ]:
print(open(os.path.join(RESULT_DIR, 'results.txt')).read())

In [ ]:
print(open(os.path.join(RESULT_DIR, 'frequency_optimisation_results.txt')).read())

In [ ]:
from IPython.display import Image, display
for fn, title in [
    ('model_comparison_events.png','Each model at its own tuned alarm policy'),
    ('tradeoff_curves.png','Operating-point trade-off curves'),
    ('roc_pr_curves.png','ROC and Precision-Recall'),
    ('well_risk_timelines.png','Predicted risk per held-out well'),
    ('per_well_performance.png','Per-well caught vs missed'),
    ('cause_and_leadtime.png','Catch rate by cause, lead-time distribution'),
    ('sensor_traces_trip.png','Sensors and model risk around a real trip'),
    ('feature_importance.png','What the detector keys on'),
    ('clustering_pca_normalized.png','Unsupervised: K-Means clusters'),
    ('cause_confusion_matrix.png','Stage 2 confusion matrix'),
]:
    try:
        print('\n' + '='*70 + '\n' + title + '\n' + '='*70)
        display(Image(os.path.join(CHART_DIR, fn)))
    except Exception as e:
        print('missing:', fn, e)

## Step 7 — Save the trained models

Freezes exactly what Step 4 trained — the selected Stage 1 detector, the Stage 2 root-cause model, the feature list and the alarm policy tuned on validation — to `model_store/esp_models.joblib`. Anything that later scores a new well loads this file and never re-fits.

Run it in the same session as Step 4: it saves the models that are in memory.

In [ ]:
import os, joblib
STORE_DIR = os.path.join(WORK, 'model_store')
os.makedirs(STORE_DIR, exist_ok=True)
joblib.dump({
    'stage1_model': models[best_name], 'stage1_name': best_name,
    'stage2_model': cause_model, 'stage2_label_encoder': le,
    'all_features': ALL_FEATURES, 'stage2_features': STAGE2_FEATURES,
    'equip_features': EQUIP_FEATURES, 'sensor_cols': SENSOR_COLS,
    'global_std_floor': global_std_floor, 'horizon_min': H,
    'alarm_quantile': Q, 'alarm_persist_min': N_PERSIST,
    'alert_window_min': ALERT_WINDOW_MIN, 'min_useful_lead_min': MIN_USEFUL_LEAD_MIN,
    'calib_days': 30,
    'trained_on_wells': [int(w) for w in TRAIN_WELLS],
    'held_out_wells': [int(w) for w in TEST_WELLS],
}, f'{STORE_DIR}/esp_models.joblib', compress=3)
print(f'Froze {best_name} + root-cause model -> {STORE_DIR}/esp_models.joblib')
print(f'Alarm policy: top {100*(1-Q):.1f}% of each well\'s own scores, {N_PERSIST} consecutive minutes')

## Step 8 — Live dashboard for a brand-new well (local VS Code run)

Simulates **well 17** — a well the models have never seen — scores it minute by minute with the models frozen in Step 7, and writes `6_dashboard/ESP_Live_Dashboard.html`: a control-room page that replays the well as a live SCADA feed at any speed. Nothing is re-trained.

On an underload alarm it also gives **frequency advice** from Step 5's method: the well's drawdown coefficient k is fitted by linear regression on its first 30 days (`Pi = a − k·Hz`), the advised frequency is the cut that recovers the intake-pressure deficit, clipped to the equipment's safe band, and the break-even downtime from Step 5 says whether acting on it pays.

Needs the project's `6_dashboard/` folder, so it runs from the project, not from a bare Colab upload. ~30 s.

In [ ]:
import webbrowser
DASH = os.path.join(PROJECT, "6_dashboard")
if not os.path.isdir(DASH):
    print("6_dashboard/ not found next to this notebook - run it from the project folder to build the dashboard.")
else:
    args = [os.path.join(DASH, "build_dashboard.py"), "--work", WORK, "--well", "17"]
    # break-even downtime per trip from Step 5 (slowing to 90% of BEP), shown next to the frequency advice
    if "be" in globals() and len(be):
        args += ["--breakeven", f"{be.loc[be['pct_of_bep'].round() == 90, 'breakeven_hours_per_trip'].median():.1f}"]
    run_script(*args)
    webbrowser.open("file:///" + os.path.join(DASH, "ESP_Live_Dashboard.html").replace(os.sep, "/"))

## Notes for whoever picks this up

**Where it stands.** 33 of 69 trips caught (48%), 31 of them with ≥30 minutes of warning, at 18.7 nuisance alarms per well per month and a 180-minute median lead time. Root-cause accuracy 68.6%, with GAS_LOCK and HIGH_DISCHARGE both at 1.00 precision.

**The most useful lesson in this project.** Detection was stuck at 19% and no amount of model tuning moved it. Measuring each cause's precursor strength against sensor noise showed catch rate tracked signal-to-noise almost exactly (6σ→45%, 3.1σ→43%, 0.8σ→10%, 0σ→6%). The bottleneck was the data, not the learning. Fixing the two unphysical causes lifted catch rate to 48% **and** cut alarms **and** lengthened warning — all at once, which is what happens when the signal was genuinely there.

**On the frequency work.** The control provably works — underload trips went 22 → 0 closed-loop. But it cost 19% of production to save 0.8% of uptime, so it does *not* pay under fast automatic restart. The transferable result is the break-even: slowing a well only pays if each avoided trip would otherwise keep it down longer than ~35 hours. Remote wells waiting on a crew flip the answer.

**What the sweep says and why to ignore it.** It claims production keeps rising to 120% of BEP. That is an artifact — the proxy assumes Q ∝ f without limit and penalises neither off-curve operation nor run-life loss from repeated restarts. Recommendations are capped at BEP, as is standard practice.

**Already tried and rejected** (don't redo without a new angle): frequency-detrended residual features; relative-only features (closed the val/test gap from the wrong end, catch rate fell to 14%); F1-maximised thresholds (degenerate under 1% positives); dropping vibration/voltage as *sensor channels* (gas-lock classification collapsed).

**Next steps:** cause-specific features for HIGH_TEMP, which is the one class still collapsing in Stage 2; and pointing the whole pipeline at real historical wells, which is the only thing that can settle whether the optimum sits below BEP at all.

**To add a well:** edit Step 1, re-run Steps 2→5. Step 3 only simulates the new well.